# CS781 Phase 7 — PART 2: pool-aware re-ranking, n-gram BM25, ensembles, more chunks, round-2 hard negatives, citation context, cross-encoder

Run it AFTER Part 1 (`phase7_pcr_retrieval`): it restores every finished run of Part 1 (nothing is retrained; attach Part 1's notebook output) and adds, in order of value and cheapness, what can move the score most. Part 1's best system on dev: micro-F1@7 0.3524 (the five techniques were statistically tied).
**Deadline note:** it rewrites `prediction.json` (+ `prediction.zip`) after every stage, and every stage checks a time budget first, so a stopped session still leaves the best valid file so far.

| stage | what | why | time (T4, estimate) |
|---|---|---|---|
| structure check | how the candidate pools are built; is a citation placeholder left in the query text? | in Part 1's log dev has about 974 relevant pairs for 905 non-query pool documents: nearly every candidate belongs to about ONE query | seconds |
| **POOL** | pool-aware re-ranking: a candidate that other queries of the same split match better is penalised (dual-softmax / CSLS / Sinkhorn). Settings are tuned on the TRAINING split with untrained scorers (BM25, zero-shot Phase 1), dev only confirms | possibly the biggest lever, rule-safe, cheap. On controlled synthetic structure it gives about +0.07 micro-F1@7 when popular 'hub' documents score high for every query and about +0.02 when similar queries compete; the real gain is MEASURED by the notebook on the training split and on dev | 10-15 min |
| NGRAM | BM25 on word 2-/3-/4-grams (published work on this benchmark finds n-gram BM25 variants strong); fusion weights tuned on the TRAINING split. **Hybrid track only** (lexical, not a Phase 1 model) | | 5-15 min |
| ENS + SWEEP | average all finished runs (largest ensemble within 0.003 of the best); score the best two models with 24 chunks per document instead of 12 (47% of documents are longer) | free / +0.005-0.02 | 10-25 min |
| R2 | round-2 hard negatives mined by the best trained model, 3 per query (many more negatives per query than before); two techniques retrained from the Phase 1 weights | standard dense-retrieval gain | mining 8 min + 2 x (35 + 6) min |
| CTX | if a citation placeholder is found: the sentences around it as the query | unknown until the data is checked | 10 min |
| XENC | a cross-encoder that reads a query chunk and a candidate chunk together, re-orders the top 30 | only 827 training queries: guarded, often no gain | 25 + 10 min |
| final | the chain, every step adopted only if dev micro-F1@7 rises by >= 0.005 with a paired 95% interval above 0; comparison table; rule check | | minutes |
Total about 2.5-4 hours. Pool-aware re-ranking uses only the Phase 1 models' scores of the split's own unlabeled queries (no label, no other split): it is transductive; mention it in the report and ask the TAs if unsure.

## Rules
Every counted experiment is from Part 1 (Phase 1 ModernBERT; six techniques). Part 2's additions are variants and post-processing, not new counted experiments. No outside model, weights, corpora or APIs; no test label. **prediction.json = Phase 1 systems only (`RULE_MODE = "strict"`);** the best Phase 1 + lexical hybrid is written to `prediction_hybrid.json` with its dev score. Plain BM25 is never a submission.

## Inputs
Part 1's notebook output (Add Input > Notebook Output Files), the ModernBERT Phase 1 dataset (also holds the tokenizer), Secrets `HF_TOKEN`, Internet ON, a GPU.

**Tested** on CPU with a tiny ModernBERT and synthetic data (hub-shaped files, a planted citation placeholder, a controlled score matrix for the re-ranking), including a tiny-budget session; not tested on the real data or a GPU. All timings and gains are estimates: the dev numbers printed by the notebook are the evidence.


In [ ]:
# ================= CELL 0 — CONFIG  [RUNS NOW]. Edit this cell only. =================
HF_DATASET = "Exploration-Lab/CS-781-Capstone"                      # gated Hugging Face dataset (accept its conditions first)
DATA_SUBDIR = "Phase_7_PCR"                                         # the Phase 7 folder inside the dataset repo (Train/, Dev/, Test/)
DATA_SOURCE = "hub_files"                                           # "hub_files" = download the six Phase 7 json files from the dataset repo; "files" = read them from LOCAL_DATA_DIR
LOCAL_DATA_DIR = None                                               # e.g. "/kaggle/input/my-phase7-files" (also the fallback if the download fails)
CKPT_PATHS = {"gpt2_new": None, "modernbert": None, "t5": None, "bilstm": None}   # Phase 1 final_model.pt paths; None = found automatically under /kaggle/input (only ModernBERT is used here)
TOKENIZER_PATH = None                                               # legal_bpe_32k_mask.json; None = found automatically
LSI_WORK = "/kaggle/working/pcr"                                    # results, ledger, rankings, checkpoints (the name is shared with the reused setup cell)
TOP_K = 7                                                           # the evaluator scores exactly 7 candidates per query
TIME_PROFILE = "normal"                                             # "normal" (about 4-5 h) or "fast" (about 2.5 h: shorter training caps, fewer chunks); the notebook also has a hard time budget (settings cell)
RULE_MODE = "strict"                                                # "strict": prediction.json only from Phase 1 models (+ their ensembles); the best Phase 1 + BM25 hybrid is ALSO written to prediction_hybrid.json for you to decide. "hybrid_ok": hybrids may become prediction.json. Plain BM25 is NEVER a submission.
RERUN = False                                                       # True = train runs again even if an attached earlier output already holds them
SELFTEST = False
NB_T0 = __import__("time").time()                                   # start of the time budget


In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
GITHUB_TOKEN = user_secrets.get_secret("GITHUB_TOKEN")
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
KAGGLE_KEY = user_secrets.get_secret("KAGGLE_KEY")
KAGGLE_USERNAME = user_secrets.get_secret("KAGGLE_USERNAME")

In [ ]:
# ================= CELL 1 — setup (Phase 7: Prior Case Retrieval, Part 2) =================
import os, sys, re, json, math, time, random, glob, gc, copy, hashlib, pickle, getpass, zipfile, shutil
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F

SEED = 42
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); AMP = device.type == "cuda"
WORK = Path(LSI_WORK); (WORK / "runs").mkdir(parents=True, exist_ok=True); (WORK / "cache").mkdir(exist_ok=True); SUBDIR = WORK / "submissions"; SUBDIR.mkdir(exist_ok=True)
LEDGER = WORK / "ledger.csv"
SEARCH_ROOTS = ["/kaggle/input", "/kaggle/working"]
print("device:", torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu (training cells need a GPU)", "| work dir:", WORK)


In [ ]:
# ===== CREDENTIALS CHECK  [RUNS NOW; it never asks you to type anything]: reads HF_TOKEN (and optionally KAGGLE_USERNAME / KAGGLE_KEY) from Kaggle Secrets, verifies them, and stops early with the exact fix =====
# Needs Internet ON. Add the Secrets under Add-ons > Secrets with 'Attached' ON for THIS notebook (a committed run only sees attached secrets). The Hugging Face token is not needed when DATA_SOURCE = "files".
import os, subprocess, shutil
REQUIRE_HF, REQUIRE_KAGGLE = (DATA_SOURCE in ("hf", "hub_files")), False        # the Kaggle key is only needed for the optional backup upload (set REQUIRE_KAGGLE = True to insist)
def get_secret(name):
    v = os.environ.get(name)
    if not v:
        try:
            from kaggle_secrets import UserSecretsClient; v = UserSecretsClient().get_secret(name)
        except Exception: v = None
    return (v or "").strip().strip("'\"") or None
problems, report = [], []
hf = get_secret("HF_TOKEN"); HF_OK = False
if DATA_SOURCE not in ("hf", "hub_files"): report.append("Hugging Face token : not needed (DATA_SOURCE = 'files')"); HF_OK = True
elif not hf: report.append("Hugging Face token : MISSING"); problems.append("HF_TOKEN is missing: add a Secret named HF_TOKEN (a token of the account that was approved for the dataset), or set DATA_SOURCE = 'files'")
else:
    try:
        from huggingface_hub import HfApi; api = HfApi(); me = api.whoami(token=hf); api.auth_check(HF_DATASET, repo_type="dataset", token=hf); HF_OK = True; os.environ["HF_TOKEN"] = hf
        report.append(f"Hugging Face token : OK (user '{me.get('name')}', access to {HF_DATASET} confirmed, {len(hf)} characters)")
    except Exception as e:
        code = getattr(getattr(e, "response", None), "status_code", None)
        why = "Hugging Face does not recognise it (wrong, expired or revoked: create a new classic 'Read' token and update the Secret)" if code == 401 else "valid token, but this account has no approved access to the gated dataset" if code == 403 else f"{type(e).__name__}: {str(e)[:100]}"
        report.append(f"Hugging Face token : REFUSED (HTTP {code}): {why}"); problems.append(f"HF_TOKEN does not work: {why}")
ku, kk = get_secret("KAGGLE_USERNAME"), get_secret("KAGGLE_KEY"); KAGGLE_OK = False
if not (ku and kk): report.append("Kaggle credentials  : not provided (the backup upload will be a dry run)"); (problems.append("KAGGLE_USERNAME / KAGGLE_KEY are missing") if REQUIRE_KAGGLE else None)
elif not shutil.which("kaggle"): report.append("Kaggle credentials  : cannot be verified (the kaggle command is not installed)"); (problems.append("the kaggle command is not installed") if REQUIRE_KAGGLE else None)
else:
    try:
        r = subprocess.run(["kaggle", "datasets", "list", "--mine"], capture_output=True, text=True, timeout=120, env={**os.environ, "KAGGLE_USERNAME": ku, "KAGGLE_KEY": kk})
        if r.returncode == 0: KAGGLE_OK = True; os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = ku, kk; report.append(f"Kaggle credentials  : OK (user '{ku}')")
        else: report.append(f"Kaggle credentials  : REFUSED ({(r.stdout + r.stderr).strip()[:100]})"); (problems.append("the Kaggle username/key were refused (use the LEGACY API key; Internet ON?)") if REQUIRE_KAGGLE else None)
    except Exception as e: report.append(f"Kaggle credentials  : not verified ({type(e).__name__})")
print("\n".join(report))
blocking = [p for p in problems if (p.startswith("HF_TOKEN") and REQUIRE_HF) or (not p.startswith("HF_TOKEN") and REQUIRE_KAGGLE)]
if blocking: raise RuntimeError("STOPPING EARLY (before any long step):\n  - " + "\n  - ".join(blocking) + "\nFix: Add-ons > Secrets > add the missing secret(s), switch 'Attached' ON for this notebook, and run again.")
print("\nCredentials ready.")


In [ ]:
# ================= CELL 4 — LOAD AND INSPECT THE PHASE 7 DATA  [RUNS NOW; CPU] =================
# Six files, each split with ITS OWN candidate pool (train.json + train_candidates.json, dev.json + dev_candidates.json, test.json + test_candidates.json). The schema is DETECTED, not assumed: candidate files may be a dict id -> text or a list of records;
# text may be a string or a list of sentences; labels may be strings or numbers (matched to the pool's ids, so zero padding is preserved). Test labels, if a test file carries any, are IGNORED.
FILES = {"train": ("train.json", "train_candidates.json"), "dev": ("dev.json", "dev_candidates.json"), "test": ("test.json", "test_candidates.json")}
def _find(name, roots):
    hits = [p for r in roots for p in glob.glob(f"{r}/**/*", recursive=True) if os.path.basename(p).lower() == name.lower() and "phase" in p.lower() and "7" in p]
    return min(hits, key=len) if hits else None
roots = [r for r in ([LOCAL_DATA_DIR] if LOCAL_DATA_DIR else []) + ["/kaggle/input", str(WORK / "hf")] if r]
if DATA_SOURCE == "hub_files" and not all(_find(f, roots) for pair in FILES.values() for f in pair):
    from huggingface_hub import snapshot_download
    snapshot_download(HF_DATASET, repo_type="dataset", allow_patterns=[f"{DATA_SUBDIR}/*", f"{DATA_SUBDIR}/**"], token=os.environ.get("HF_TOKEN") or get_secret("HF_TOKEN"), local_dir=str(WORK / "hf")); roots.append(str(WORK / "hf"))
PATHS = {s: tuple(_find(f, roots) for f in pair) for s, pair in FILES.items()}; missing = [f for s, pair in FILES.items() for f, p in zip(pair, PATHS[s]) if not p]
assert not missing, f"Phase 7 files not found: {missing} (searched {roots}); attach them or check DATA_SUBDIR / LOCAL_DATA_DIR"
def _load(p):
    t = open(p, encoding="utf-8").read().strip()
    try: return json.loads(t)
    except json.JSONDecodeError: return [json.loads(l) for l in t.splitlines() if l.strip()]
def _text(v): return v if isinstance(v, str) else " ".join(_text(x) for x in v) if isinstance(v, (list, tuple)) else " ".join(_text(x) for x in v.values()) if isinstance(v, dict) else str(v)
ID_KEYS = ("id", "query_id", "case_id", "candidate_id", "doc_id", "qid", "cid"); TEXT_KEYS = ("text", "sentences", "document", "content", "body", "judgment", "doc")
LAB_KEYS = ("relevant_candidates", "relevant_candidate_ids", "relevant", "relevant_ids", "labels", "label", "positives", "gold", "citations", "cited", "relevant_cases")
def _records(obj):
    if isinstance(obj, dict):
        for k in ("data", "queries", "candidates", "records", "items"):
            if isinstance(obj.get(k), list): return obj[k], None
        vals = list(obj.values())[:20]
        if vals and all(isinstance(v, (str, list)) for v in vals): return [{"id": k, "text": v} for k, v in obj.items()], "dict id -> text"
        if vals and all(isinstance(v, dict) for v in vals): return [{"id": k, **v} for k, v in obj.items()], "dict id -> record"
        raise ValueError("unrecognised json object layout")
    return obj, None
def parse(obj, with_labels):
    recs, how = _records(obj); r0 = recs[0]; ik = next(k for k in ID_KEYS if k in r0); tk = next(k for k in TEXT_KEYS if k in r0); lk = next((k for k in LAB_KEYS if k in r0), None) if with_labels else None
    return [r[ik] for r in recs], [_text(r[tk]) for r in recs], ([r.get(lk) or [] for r in recs] if lk else None), dict(id=ik, text=tk, labels=lk, layout=how or "list of records")
DATA, SCHEMA = {}, {}
for s, (qp, cp) in PATHS.items():
    q_raw, q_text, q_lab, qs = parse(_load(qp), with_labels=(s != "test")); c_raw, c_text, _, cs = parse(_load(cp), with_labels=False)
    if s == "test" and any(k in _records(_load(qp))[0][0] for k in LAB_KEYS): print("[test] NOTE: test.json carries a relevance field. Test labels are private, so it is IGNORED and never read.")
    c_ids = [str(x) for x in c_raw]; by_int = {}
    for c in c_ids:
        if c.isdigit(): by_int.setdefault(int(c), c)
    def norm(lbl):
        out = []
        for x in (lbl if isinstance(lbl, (list, tuple, set)) else [lbl]):
            x = str(x)
            if x in pos_c: out.append(x)
            elif x.isdigit() and int(x) in by_int: out.append(by_int[int(x)])
            else: out.append(None)
        return out
    pos_c = {c: i for i, c in enumerate(c_ids)}; gold = None; lost = 0
    if q_lab is not None:
        gold = []
        for l in q_lab: n_ = norm(l); lost += sum(x is None for x in n_); gold.append({pos_c[x] for x in n_ if x is not None})
    DATA[s] = dict(q_ids=[str(x) for x in q_raw], q_raw=q_raw, q_text=q_text, gold=gold, c_ids=c_ids, c_raw=c_raw, c_text=c_text, c_pos=pos_c, lost_labels=lost); SCHEMA[s] = dict(queries=qs, candidates=cs)
print("SCHEMA (detected):", json.dumps(SCHEMA, indent=None)[:600])
rows = []
for s, D in DATA.items():
    wl_q, wl_c = [len(t.split()) for t in D["q_text"]], [len(t.split()) for t in D["c_text"]]; self_in = sum(q in D["c_pos"] for q in D["q_ids"])
    g = [len(x) for x in D["gold"]] if D["gold"] else []
    rows.append(dict(split=s, queries=len(D["q_ids"]), candidates=len(D["c_ids"]), query_words_p50=int(np.median(wl_q)), cand_words_p50=int(np.median(wl_c)), cand_words_p90=int(np.percentile(wl_c, 90)), gold_mean=round(float(np.mean(g)), 2) if g else None,
                     gold_p50=float(np.median(g)) if g else None, gold_max=max(g) if g else None, no_gold=sum(x == 0 for x in g) if g else None, labels_not_in_pool=D["lost_labels"], queries_inside_own_pool=self_in, dup_cand_ids=len(D["c_ids"]) - len(set(D["c_ids"]))))
print(pd.DataFrame(rows).to_string(index=False))
pools = {s: set(D["c_ids"]) for s, D in DATA.items()}; print("candidate-pool overlap between splits:", {f"{a}&{b}": len(pools[a] & pools[b]) for a, b in (("train", "dev"), ("train", "test"), ("dev", "test"))}, "| every split is retrieved ONLY from its own pool")
GOLD_MEAN = float(np.mean([len(x) for x in DATA["dev"]["gold"]])) if DATA["dev"]["gold"] else None; print(f"dev: a query has {GOLD_MEAN:.2f} relevant candidates on average; the evaluator compares the top {TOP_K} with the WHOLE gold set (micro-F1@{TOP_K})")


In [ ]:
# ================= CELL 5 — PHASE 1 TOKENIZER AND CHUNKS  [RUNS NOW; CPU, a few minutes] =================
# Every document keeps its token ids; chunks (512 tokens, spread EVENLY over the document) are cut on demand, so the number of chunks can be changed without tokenising again (MAX_CHUNKS for training / default scoring, MAX_CHUNKS_BIG for the chunk sweep).
from tokenizers import Tokenizer
tpaths = [TOKENIZER_PATH] if TOKENIZER_PATH else sorted(glob.glob("/kaggle/input/**/legal_bpe_32k_mask.json", recursive=True)) or sorted(glob.glob("/kaggle/input/**/legal_bpe_32k*.json", recursive=True))
assert tpaths, "the Phase 1 tokenizer (legal_bpe_32k_mask.json) was not found: attach a Phase 1 dataset"
tok = Tokenizer.from_file(tpaths[0]); VOCAB = tok.get_vocab_size(); PAD = tok.token_to_id("[PAD]") or 0; BOS, EOS = tok.token_to_id("[BOS]"), tok.token_to_id("[EOS]")
CHUNK_TOK, MAX_CHUNKS, MAX_CHUNKS_BIG = 512, 12, 24
if SELFTEST: CHUNK_TOK, MAX_CHUNKS, MAX_CHUNKS_BIG = 48, 3, 6
def chunk_doc(ids, max_chunks=None):
    body = CHUNK_TOK - 2; n = max(1, math.ceil(len(ids) / body)); m = max_chunks or MAX_CHUNKS; pick = sorted(set(np.linspace(0, n - 1, min(n, m)).round().astype(int).tolist())); out = []
    for k in pick:
        seg = list(ids[k * body:(k + 1) * body])
        if seg or k == 0: out.append(np.asarray(([BOS] if BOS is not None else []) + seg + ([EOS] if EOS is not None else []), dtype=np.int32))
    return out
t0 = time.time(); TOK_LEN = {}
for s, D in DATA.items():
    for side in ("q", "c"):
        encs = tok.encode_batch(D[f"{side}_text"], add_special_tokens=False); D[f"{side}_tok"] = [np.asarray(e.ids, dtype=np.int32) for e in encs]; D[f"{side}_chunks"] = [chunk_doc(i) for i in D[f"{side}_tok"]]; TOK_LEN[(s, side)] = [len(i) for i in D[f"{side}_tok"]]; del encs
_CH_CACHE = {}
def chunks_for(s, side, m=None):
    m = m or MAX_CHUNKS
    if m == MAX_CHUNKS: return DATA[s][f"{side}_chunks"]
    if (s, side, m) not in _CH_CACHE: _CH_CACHE[(s, side, m)] = [chunk_doc(i, m) for i in DATA[s][f"{side}_tok"]]
    return _CH_CACHE[(s, side, m)]
print(f"tokenised every document in {time.time() - t0:.0f}s | chunk {CHUNK_TOK} tokens, {MAX_CHUNKS} chunks by default ({MAX_CHUNKS_BIG} in the sweep), vocabulary {VOCAB}")
print(pd.DataFrame([dict(split=s, side={"q": "queries", "c": "candidates"}[side], tokens_p50=int(np.median(v)), tokens_p90=int(np.percentile(v, 90)), longer_than_default_chunks=f"{np.mean(np.asarray(v) > MAX_CHUNKS * (CHUNK_TOK - 2)):.0%}", longer_than_big=f"{np.mean(np.asarray(v) > MAX_CHUNKS_BIG * (CHUNK_TOK - 2)):.0%}") for (s, side), v in TOK_LEN.items()]).to_string(index=False))


In [ ]:
# ================= CELL 8 — LOAD THE PHASE 1 MODELS  [NEEDS the Phase 1 checkpoints; set CKPT_PATHS in the config cell or attach the datasets]. The old 3,000-step GPT-2 is never used. =================
ARCH_LOADERS_SRC = r'''"""arch_loaders.py - load any of the four pre-trained architectures as a backbone for classification / tagging.
   load_backbone(path) -> (backbone, hidden_size, causal, max_len, family)
   backbone(input_ids, attention_mask) -> object with .last_hidden_state of shape [B, T, hidden_size]"""
import torch, torch.nn as nn
from types import SimpleNamespace
from transformers import GPT2Config, GPT2LMHeadModel, ModernBertConfig, ModernBertForMaskedLM, T5Config, T5ForConditionalGeneration

class BiLSTMLM(nn.Module):
    def __init__(self, vocab, emb=512, hidden=1024, layers=3):
        super().__init__()
        self.vocab, self.emb_dim, self.hidden, self.layers = vocab, emb, hidden, layers
        self.emb = nn.Embedding(vocab, emb); self.fwd = nn.LSTM(emb, hidden, layers, batch_first=True); self.bwd = nn.LSTM(emb, hidden, layers, batch_first=True)
        self.norm_f, self.norm_b = nn.LayerNorm(hidden), nn.LayerNorm(hidden)
        self.proj_f, self.proj_b = nn.Linear(hidden, emb), nn.Linear(hidden, emb); self.out_bias = nn.Parameter(torch.zeros(vocab))
    def hidden_states(self, input_ids, attention_mask=None):
        x = input_ids; B, T = x.shape
        lengths = attention_mask.sum(1) if attention_mask is not None else torch.full((B,), T, device=x.device)
        rev = (lengths[:, None] - 1 - torch.arange(T, device=x.device)[None, :]).clamp(min=0)
        hf, _ = self.fwd(self.emb(x)); hb, _ = self.bwd(self.emb(x.gather(1, rev)))
        hb = hb.gather(1, rev[:, :, None].expand(-1, -1, hb.shape[-1]))
        return torch.cat([hf, hb], dim=-1)

class Backbone(nn.Module):
    """one calling convention for every family (matches what the Phase 3 SeqClassifier expects)"""
    def __init__(self, family, inner, embed):
        super().__init__(); self.family, self.inner, self._embed = family, inner, embed
    def get_input_embeddings(self): return self._embed
    def forward(self, input_ids=None, attention_mask=None, inputs_embeds=None, use_cache=False, **kw):
        f = self.family
        if f == "gpt2": h = self.inner(input_ids=input_ids, attention_mask=attention_mask, inputs_embeds=inputs_embeds, use_cache=False).last_hidden_state
        else:
            assert inputs_embeds is None, f"prompt tuning (inputs_embeds) is only wired for the gpt2 family, not {f}"
            if f in ("modernbert", "t5"): h = self.inner(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
            elif f == "bilstm": h = self.inner.hidden_states(input_ids, attention_mask)
            else: raise ValueError(f)
        return SimpleNamespace(last_hidden_state=h)

CAUSAL = {"gpt2": True, "modernbert": False, "t5": False, "bilstm": False}
# modules to adapt with LoRA / LoRMA / IA3 (None = full fine-tuning only). ModernBERT's q,k,v are ONE fused linear, 'Wqkv' (rows: q | k | v).
ADAPTER_TARGETS = {"gpt2": "attn.c_attn (fused Conv1D: q|k|v)", "modernbert": "attn.Wqkv (fused Linear: q|k|v)", "t5": "SelfAttention q, k, v, o (Linear)", "bilstm": None}

def build_full(mc):
    f, V, d = mc["family"], mc["vocab_size"], mc["hidden_size"]
    if f == "gpt2":
        return GPT2LMHeadModel(GPT2Config(vocab_size=V, n_positions=mc["context_length"], n_ctx=mc["context_length"], n_embd=d, n_layer=mc["num_layers"], n_head=mc["num_heads"], n_inner=mc.get("n_inner", 4 * d),
                                          resid_pdrop=0.0, embd_pdrop=0.0, attn_pdrop=0.0))
    if f == "modernbert":
        return ModernBertForMaskedLM(ModernBertConfig(vocab_size=V, max_position_embeddings=mc["context_length"], hidden_size=d, intermediate_size=mc["intermediate_size"], num_hidden_layers=mc["num_layers"],
                                                      num_attention_heads=mc["num_heads"], local_attention=mc["local_attention"], global_attn_every_n_layers=mc["global_attn_every_n_layers"], pad_token_id=0,
                                                      attention_dropout=0.0, mlp_dropout=0.0, embedding_dropout=0.0, classifier_pooling="mean", sparse_prediction=True, sparse_pred_ignore_index=-100, reference_compile=False))
    if f == "t5":
        return T5ForConditionalGeneration(T5Config(vocab_size=V, d_model=d, d_kv=mc["d_kv"], d_ff=mc["d_ff"], num_layers=mc["num_layers"], num_decoder_layers=mc["num_decoder_layers"], num_heads=mc["num_heads"],
                                                   feed_forward_proj="gated-gelu", dropout_rate=0.0, tie_word_embeddings=True))
    if f == "bilstm": return BiLSTMLM(V, emb=mc["emb"], hidden=mc["lstm_hidden"], layers=mc["num_layers"])
    raise ValueError(f)

def load_backbone(path, device="cpu"):
    ck = torch.load(path, map_location="cpu", weights_only=False); mc = ck["model_config"]; full = build_full(mc); full.load_state_dict(ck["model_state_dict"], strict=True)
    f = mc["family"]
    if f == "gpt2": inner, embed = full.transformer, full.transformer.wte
    elif f == "modernbert": inner, embed = full.model, full.model.embeddings.tok_embeddings
    elif f == "t5": inner, embed = full.encoder, full.shared
    elif f == "bilstm": inner, embed = full, full.emb
    else: raise ValueError(f)
    return Backbone(f, inner, embed).to(device), mc["hidden_size"], CAUSAL[f], mc["context_length"], f
'''
al_paths = sorted(glob.glob("/kaggle/input/**/arch_loaders.py", recursive=True)) or sorted(glob.glob("/kaggle/working/**/arch_loaders.py", recursive=True))
if not al_paths:                                                                  # self-contained: the loader module is embedded in this notebook
    (WORK / "arch_loaders.py").write_text(ARCH_LOADERS_SRC); al_paths = [str(WORK / "arch_loaders.py")]; print("arch_loaders.py not found among the inputs: using the copy embedded in this notebook")
sys.path.insert(0, str(Path(al_paths[0]).parent)); import arch_loaders as AL

MODEL_PATHS = {}                                    # filled automatically below; you can also set entries by hand, e.g. MODEL_PATHS["bilstm"] = "/kaggle/input/.../final_model.pt"
def discover_models():
    found = {}
    for p in sorted(glob.glob("/kaggle/input/**/*.pt", recursive=True)):
        if os.path.getsize(p) < 50e6: continue
        base = os.path.basename(p)
        if base != "final_model.pt": continue                # only the NEW Phase 1 models: the old 3,000-step checkpoint is deliberately never used
        try: ck = torch.load(p, map_location="cpu", weights_only=False, mmap=True)
        except Exception: continue
        if not isinstance(ck, dict) or "model_config" not in ck or "model_state_dict" not in ck: continue
        mc = ck["model_config"]; fam = mc.get("family")
        if fam is None: continue                              # a checkpoint without a family entry is an old-format model: skipped
        key = "gpt2_new" if fam == "gpt2" else fam
        if key not in found: found[key] = p
    return found
MODEL_PATHS.update({k: v for k, v in discover_models().items() if k not in MODEL_PATHS}); MODEL_PATHS.update({k: v for k, v in CKPT_PATHS.items() if v})        # CKPT_PATHS from the config cell wins
print("Phase 1 models found:", {k: v for k, v in MODEL_PATHS.items()}, "\n  missing (pre-train them first):", [k for k in ("gpt2_new", "modernbert", "t5", "bilstm") if k not in MODEL_PATHS])

def load_backbone_any(path):
    """like AL.load_backbone, but also accepts the OLD Phase 1 checkpoint (its model_config has no 'family' entry)"""
    ck = torch.load(path, map_location="cpu", weights_only=False); mc = dict(ck["model_config"]); mc.setdefault("family", "gpt2")
    full = AL.build_full(mc); full.load_state_dict(ck["model_state_dict"], strict=True); f = mc["family"]
    if f == "gpt2": inner, embed = full.transformer, full.transformer.wte
    elif f == "modernbert": inner, embed = full.model, full.model.embeddings.tok_embeddings
    elif f == "t5": inner, embed = full.encoder, full.shared
    elif f == "bilstm": inner, embed = full, full.emb
    else: raise ValueError(f)
    del ck; return AL.Backbone(f, inner, embed), mc["hidden_size"], AL.CAUSAL[f], mc["context_length"], f

def set_backbone_dropout(backbone, p):
    """turn on dropout inside the backbone for fine-tuning (works where the model has nn.Dropout modules, e.g. GPT-2 and T5; a no-op for the others). Returns how many modules were changed."""
    n = 0
    for mod in backbone.modules():
        if isinstance(mod, nn.Dropout): mod.p = p; n += 1
    return n

def truncate_layers(bb, n):
    """keep only the first n transformer layers (initialises a smaller student from the same Phase 1 weights); implemented for the GPT-2 family"""
    assert bb.family == "gpt2", "layer truncation is implemented for the GPT-2 family only"
    bb.inner.h = nn.ModuleList(list(bb.inner.h)[:n]); bb.inner.config.n_layer = n; return bb

def refresh_models(verbose=True):
    """re-scan /kaggle/input for Phase 1 models (for example after you attached another dataset to the running session)"""
    new = {k: v for k, v in discover_models().items() if k not in MODEL_PATHS}; MODEL_PATHS.update(new)
    if verbose and new: print("newly found Phase 1 models:", new)
    return sorted(MODEL_PATHS)

class MLClassifier(nn.Module):
    """Phase 1 backbone + a multi-label head. One window of <= L tokens in, one logit per statute out. Causal models (GPT-2) pool the last token, the others the mean of the tokens."""
    def __init__(self, backbone, hidden, causal, n_labels, dropout=0.1, pool="last"):
        super().__init__(); self.backbone, self.hidden, self.causal, self.n_labels, self.pool = backbone, hidden, causal, n_labels, pool
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden, hidden), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden, n_labels))
    def forward(self, ids, mask):
        T = int(mask.sum(1).max()); ids, mask = ids[:, :T], mask[:, :T]
        h = self.backbone(input_ids=ids, attention_mask=mask).last_hidden_state; mm = mask.unsqueeze(-1).to(h.dtype); mean = (h * mm).sum(1) / mm.sum(1).clamp(min=1)
        if self.causal:
            last = h[torch.arange(h.shape[0], device=h.device), mask.sum(1) - 1]; pooled = last if self.pool == "last" else 0.5 * (last + mean)
        else: pooled = mean
        return self.head(pooled)

# ---- windows over the stored tokens of a case (the whole case if <= 8,192 tokens, else its first 4,096 + last 4,096) ----
def window_spans(n, L, W, order):
    """up to W (start, end) spans of <= L tokens. order: 'head' (from the start), 'tail' (from the end), 'both' (head and tail windows interleaved, skipping a window already covered)"""
    nb = max(1, -(-n // L)); head = [(k * L, min(n, (k + 1) * L)) for k in range(nb)]; tail = [(max(0, n - (k + 1) * L), n - k * L) for k in range(nb)]
    if order == "head": sp = head
    elif order == "tail": sp = tail
    else:
        sp = []
        for a, b in zip(head, tail):
            for s_ in (a, b):
                if not any(s_[0] >= c[0] and s_[1] <= c[1] for c in sp): sp.append(s_)
    return sp[:W]
def build_batch(toks, idx, spans, L):
    ids = np.full((len(idx), L), PAD, dtype=np.int64); mask = np.zeros((len(idx), L), dtype=np.int64)
    for b, (i, (s, e)) in enumerate(zip(idx, spans)):
        seg = toks[i][s:e]; ids[b, :len(seg)] = seg; mask[b, :len(seg)] = 1
    return torch.from_numpy(ids), torch.from_numpy(mask)


In [ ]:
# ================= CELL 9 — ADAPTATION TECHNIQUES  [pure code]: full, partial (layer-wise freezing), LoRA, rsLoRA, LoRMA+, LoRMA-pi, IA3 =================
from transformers.pytorch_utils import Conv1D
class LoRADelta(nn.Module):                # additive:  dW x = (alpha/r) B A x   (rsLoRA: alpha/sqrt(r))
    def __init__(self, nx, nf, r, alpha, drop, rs=False):
        super().__init__()
        self.A = nn.Parameter(torch.empty(nx, r).uniform_(-1 / math.sqrt(nx), 1 / math.sqrt(nx))); self.B = nn.Parameter(torch.zeros(r, nf))
        self.scale = alpha / (math.sqrt(r) if rs else r); self.drop = nn.Dropout(drop)
    def forward(self, x): return (self.drop(x) @ self.A) @ self.B * self.scale
class LoRMAPlus(nn.Module):                # multiplicative, additive rank inflation: h' = (I + (alpha/r) B A) h ; B = 0 at init  (= W0 x + (alpha/r) B A (W0 x))
    def __init__(self, d, r, alpha):
        super().__init__()
        self.A = nn.Parameter(torch.empty(r, d).uniform_(-1 / math.sqrt(d), 1 / math.sqrt(d))); self.B = nn.Parameter(torch.zeros(d, r)); self.scale = alpha / r
    def forward(self, h): return h + ((h @ self.A.t()) @ self.B.t()) * self.scale
class LoRMAPi(nn.Module):                  # multiplicative, permutation-based rank inflation: h' = I_pi(BA) h, I_pi(BA) = I at init
    def __init__(self, d, r):
        super().__init__()
        B = torch.empty(d, r).uniform_(-1 / math.sqrt(d), 1 / math.sqrt(d)); B[:, 0] = 1.0; A = torch.zeros(r, d); A[0, 0] = 1.0
        self.B, self.A = nn.Parameter(B), nn.Parameter(A); i = torch.arange(d); self.register_buffer("idx", (i[None, :] - i[:, None]) % d)
    def matrix(self): return (self.B @ self.A).gather(1, self.idx)
    def forward(self, h): return h @ self.matrix().t().to(h.dtype)
class _Wrap:
    """expose the wrapped layer's weight / bias: some Hugging Face modules read them directly (T5's feed-forward dtype check does)"""
    @property
    def weight(self): return self.base.weight
    @property
    def bias(self): return self.base.bias
def proj_dims(m):
    return (m.in_features, m.out_features) if isinstance(m, nn.Linear) else tuple(m.weight.shape)      # HF Conv1D weight is [in, out]
class FusedQV(_Wrap, nn.Module):                  # wraps a FUSED q|k|v projection (GPT-2 Conv1D c_attn, ModernBERT Linear Wqkv); LoRA / LoRMA adapt q and v, IA3 rescales k and v
    def __init__(self, base, kind, r=8, alpha=16, drop=0.05, rs=False):
        super().__init__(); nx, nf = proj_dims(base); d = nf // 3; self.base, self.kind, self.d = base, kind, d
        if kind == "lora": self.q, self.v = LoRADelta(nx, d, r, alpha, drop, rs), LoRADelta(nx, d, r, alpha, drop, rs)
        elif kind == "lorma_plus": self.q, self.v = LoRMAPlus(d, r, alpha), LoRMAPlus(d, r, alpha)
        elif kind == "lorma_pi": self.q, self.v = LoRMAPi(d, r), LoRMAPi(d, r)
        elif kind == "ia3": self.lk, self.lv = nn.Parameter(torch.ones(d)), nn.Parameter(torch.ones(d))
        else: raise ValueError(kind)
    def forward(self, x):
        out = self.base(x); d = self.d; q, k, v = out[..., :d], out[..., d:2 * d], out[..., 2 * d:]
        if self.kind == "lora": q, v = q + self.q(x), v + self.v(x)
        elif self.kind.startswith("lorma"): q, v = self.q(q), self.v(v)
        else: k, v = k * self.lk, v * self.lv
        return torch.cat([q, k, v], -1)
class IA3Input(_Wrap, nn.Module):                 # IA3 on the FFN: rescale the activation entering the down-projection
    def __init__(self, base): super().__init__(); self.base = base; self.l = nn.Parameter(torch.ones(proj_dims(base)[0]))
    def forward(self, x): return self.base(x * self.l)
class LinAdapt(_Wrap, nn.Module):                 # the same adapters for separate nn.Linear projections (T5's q, k, v)
    def __init__(self, base, kind, r=8, alpha=16, drop=0.05, rs=False):
        super().__init__(); self.base, self.kind = base, kind; di, do = base.in_features, base.out_features
        if kind == "lora": self.op = LoRADelta(di, do, r, alpha, drop, rs)
        elif kind == "lorma_plus": self.op = LoRMAPlus(do, r, alpha)
        elif kind == "lorma_pi": self.op = LoRMAPi(do, r)
        elif kind == "ia3": self.l = nn.Parameter(torch.ones(do))
    def forward(self, x):
        y = self.base(x)
        if self.kind == "lora": return y + self.op(x)
        if self.kind == "ia3": return y * self.l
        return self.op(y)
def wrap_linear_targets(root, kind, names, **kw):
    n = 0
    for _, parent in list(root.named_modules()):
        for cname, child in list(parent.named_children()):
            if isinstance(child, nn.Linear) and cname in names: setattr(parent, cname, LinAdapt(child, kind, **kw)); n += 1
    assert n > 0, f"no nn.Linear named {names} found in the backbone"
def freeze_lower(model, frac=0.5):
    """partial / layer-wise fine-tuning: freeze the embeddings and the lowest `frac` of the layers, train the rest"""
    inner, fam = model.backbone.inner, model.backbone.family
    for p in inner.parameters(): p.requires_grad = True
    if fam == "bilstm":
        n = inner.layers; k = int(n * frac)
        for p in inner.emb.parameters(): p.requires_grad = False
        for name, p in list(inner.fwd.named_parameters()) + list(inner.bwd.named_parameters()):
            if int(name.split("_l")[-1]) < k: p.requires_grad = False
        return
    if fam == "gpt2": layers, pre = list(inner.h), [inner.wte, inner.wpe]
    elif fam == "modernbert": layers, pre = list(inner.layers), [inner.embeddings]
    elif fam == "t5": layers, pre = list(inner.block), [inner.embed_tokens]
    else: raise ValueError(fam)
    k = int(len(layers) * frac)
    for mod in pre:
        for p in mod.parameters(): p.requires_grad = False
    for lay in layers[:k]:
        for p in lay.parameters(): p.requires_grad = False
METHODS = ("full", "partial", "lora", "rslora", "lorma_plus", "lorma_pi", "ia3")
ADAPTER_LR = {"lora": 5e-4, "rslora": 5e-4, "lorma_plus": 5e-4, "lorma_pi": 2e-4, "ia3": 3e-3}        # starting points; use lr_pilot(..., method=...) to check them
def apply_method(model, method, hp=None):
    assert method in METHODS, method; hp = hp or {}
    if method == "full":
        for p in model.backbone.parameters(): p.requires_grad = True
        return model
    if method == "partial": freeze_lower(model, hp.get("frac", 0.5)); return model
    fam, inner = model.backbone.family, model.backbone.inner
    for p in model.backbone.parameters(): p.requires_grad = False
    kind = {"lora": "lora", "rslora": "lora", "lorma_plus": "lorma_plus", "lorma_pi": "lorma_pi", "ia3": "ia3"}[method]; kw = dict(r=hp.get("r", 16), alpha=hp.get("alpha", 32), drop=hp.get("adapter_dropout", 0.05), rs=(method == "rslora"))
    if fam == "gpt2":
        for blk in inner.h:
            assert isinstance(blk.attn.c_attn, Conv1D), "unexpected c_attn type in this transformers version"; blk.attn.c_attn = FusedQV(blk.attn.c_attn, kind, **kw)
            if kind == "ia3": blk.mlp.c_proj = IA3Input(blk.mlp.c_proj)
    elif fam == "modernbert":
        for lay in inner.layers:
            assert isinstance(lay.attn.Wqkv, nn.Linear); lay.attn.Wqkv = FusedQV(lay.attn.Wqkv, kind, **kw)
            if kind == "ia3": lay.mlp.Wo = IA3Input(lay.mlp.Wo)
    elif fam == "t5":
        wrap_linear_targets(inner, kind, ("k", "v") if kind == "ia3" else ("q", "v"), **({} if kind == "ia3" else kw))
        if kind == "ia3":
            for _, parent in list(inner.named_modules()):
                if "DenseGatedActDense" in type(parent).__name__ or "DenseReluDense" in type(parent).__name__: parent.wo = IA3Input(parent.wo)
    else: raise NotImplementedError(f"'{method}' needs attention/linear layers: the {fam} model supports only 'full' and 'partial'")
    return model


In [ ]:
# ================= CELL 8 — METRICS, RANKING AND THE SUBMISSION FILE  [pure code] =================
# micro-P@7 = sum of hits / (7 x queries); micro-R@7 = sum of hits / sum of gold sizes (the WHOLE gold set, neither truncated nor padded); micro-F1@7 = their harmonic mean.
def rank_top(scores, D, k=TOP_K):
    """scores [n_queries, n_candidates] -> for every query the indices of its top-k candidates (the query itself is never returned if it sits in its own pool)"""
    S = np.array(scores, dtype=np.float32, copy=True)
    for qi, q in enumerate(D["q_ids"]):
        if q in D["c_pos"]: S[qi, D["c_pos"][q]] = -np.inf
    return np.argsort(-S, axis=1)[:, :k]
def micro_at_k(top, gold, k=TOP_K):
    hits = np.array([len(set(t[:k].tolist()) & g) for t, g in zip(top, gold)], dtype=float); gs = np.array([len(g) for g in gold], dtype=float)
    p = hits.sum() / (k * len(gold)); r = hits.sum() / max(1.0, gs.sum()); f = 0.0 if p + r == 0 else 2 * p * r / (p + r); return dict(f1=f, p=p, r=r, hits=hits, gsize=gs)
def boot_diff(m_a, m_b, n=1000, seed=0, k=TOP_K):
    """paired bootstrap over QUERIES of the micro-F1@k difference a - b: (mean, 2.5%, 97.5%)"""
    r = np.random.default_rng(seed); ha, hb, gs = m_a["hits"], m_b["hits"], m_a["gsize"]; nq = len(gs); out = []
    for _ in range(n):
        i = r.integers(0, nq, nq); f = lambda h: (lambda p, rr: 0.0 if p + rr == 0 else 2 * p * rr / (p + rr))(h[i].sum() / (k * nq), h[i].sum() / max(1.0, gs[i].sum())); out.append(f(ha) - f(hb))
    out = np.array(out); return float(out.mean()), float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5))
def write_prediction(path, top, D):
    sub = {D["q_raw"][qi] if isinstance(D["q_raw"][qi], str) else str(D["q_raw"][qi]): [D["c_raw"][ci] if isinstance(D["c_raw"][ci], str) else str(D["c_raw"][ci]) for ci in top[qi][:TOP_K]] for qi in range(len(D["q_ids"]))}
    json.dump(sub, open(path, "w", encoding="utf-8"), ensure_ascii=False); return sub
def validate_prediction(path, D):
    sub = json.load(open(path, encoding="utf-8")); pool = set(map(str, D["c_raw"])); qs = list(map(str, D["q_raw"]))
    assert isinstance(sub, dict), "prediction.json must be a JSON object: query id -> list of candidate ids"
    assert set(sub) == set(qs) and len(sub) == len(qs), f"query ids differ from the test file ({len(set(sub) ^ set(qs))} differ)"
    for q, lst in sub.items():
        assert isinstance(lst, list) and len(lst) == TOP_K, f"{q}: {len(lst)} candidates instead of {TOP_K}"
        assert len(set(lst)) == TOP_K, f"{q}: duplicate candidates"; assert all(c in pool for c in lst), f"{q}: a candidate outside the TEST pool"; assert q not in lst, f"{q}: the query itself was returned"
    return True
def zip_prediction(path, zpath):
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z: z.write(path, arcname="prediction.json")          # the file at the ROOT of the archive, no folder


In [ ]:
# ================= CELL 12 — THE ROLLING FINAL FILE  [pure code] =================
# After EVERY finished run the best RULE-SAFE system so far (by dev micro-F1@7) writes prediction.json + prediction.zip, so a valid file always exists even if the session stops early.
CKPT = WORK / "checkpoints"; CKPT.mkdir(parents=True, exist_ok=True)
def safe(label, fn):
    try: return fn()
    except Exception as e:
        import traceback; print(f"!! {label} FAILED ({type(e).__name__}: {str(e)[:200]}); the notebook carries on"); traceback.print_exc(limit=3); return None
PRED_PATH = WORK / "prediction_SELFTEST.json" if SELFTEST else Path("/kaggle/working/prediction.json"); ZIP_PATH = PRED_PATH.with_suffix(".zip")
def runs_table():
    rows = [json.load(open(p)) for p in (WORK / "runs").glob("*/result.json")]; return pd.DataFrame(rows) if rows else pd.DataFrame()
def safe_rule(r): return r.get("technique") != "bm25" and not str(r.get("name", "")).startswith("H_")          # rule-safe = only Phase 1 models (and their ensembles)
def allowed(r):
    a = r.get("adopted", True); ok = safe_rule(r) if RULE_MODE == "strict" else r.get("technique") != "bm25"
    return ok and not (a is False or str(a) == "False")                                                  # a stage that did not pass its adoption rule is kept in the table but never becomes prediction.json                       # "hybrid_ok" also allows Phase 1 + BM25 hybrids; pure BM25 is NEVER a submission
def update_final(verbose=True, names=None):
    T = runs_table()
    if not len(T): return None
    T = T[[allowed(r) for r in T.to_dict("records")]]
    if names is not None: T = T[T.name.isin(names)]
    if not len(T): return None
    best = T.sort_values("dev_f1", ascending=False).iloc[0]; St = np.load(WORK / "runs" / best["name"] / "test_scores.npy").astype(np.float32)
    write_prediction(PRED_PATH, rank_top(St, DATA["test"]), DATA["test"]); validate_prediction(PRED_PATH, DATA["test"]); zip_prediction(PRED_PATH, ZIP_PATH); json.dump(dict(system=best["name"], dev_f1=float(best["dev_f1"])), open(WORK / "final_choice.json", "w"))
    if verbose: print(f"prediction.json <- {best['name']} (dev micro-F1@7 {best['dev_f1']:.4f}) | validated: {len(DATA['test']['q_ids'])} test queries x {TOP_K} unique candidates from the TEST pool | zip: {ZIP_PATH.name}")
    return best["name"]


In [ ]:
# ================= CELL 10 — THE RETRIEVAL ENCODER, ENCODING AND SCORING  [pure code] =================
# A BI-ENCODER on the Phase 1 ModernBERT: every 512-token chunk -> the mean of its token states -> a unit vector. Two ways to score a query against a candidate (both measured on dev; the better one is used per run):
#   'mean'   cosine of the mean chunk vectors (a document-level representation)
#   'maxsim' for every query chunk the best-matching candidate chunk, averaged over the query chunks (chunk-level, keeps WHERE the documents agree)
def truncate_modernbert(bb, n):
    bb.inner.layers = nn.ModuleList(list(bb.inner.layers)[:n]); bb.inner.config.num_hidden_layers = n; return bb
class DocEncoder(nn.Module):
    def __init__(self, bb, hidden, n_prompt=0):
        super().__init__(); self.backbone, self.hidden, self.n_prompt = bb, hidden, n_prompt
        if n_prompt:                                                             # prompt tuning: n_prompt trained vectors in front of every chunk, initialised from real token embeddings
            E = bb.inner.embeddings.tok_embeddings.weight; self.prompt = nn.Parameter(E[torch.randint(10, E.shape[0], (n_prompt,))].detach().clone())
    def forward(self, ids, mask):
        T = int(mask.sum(1).max()); ids, mask = ids[:, :T], mask[:, :T]
        if self.n_prompt:
            e = self.backbone.inner.embeddings.tok_embeddings(ids); P = self.prompt.to(e.dtype).unsqueeze(0).expand(ids.shape[0], -1, -1)
            h = self.backbone.inner(inputs_embeds=torch.cat([P, e], 1), attention_mask=torch.cat([torch.ones(ids.shape[0], self.n_prompt, dtype=mask.dtype, device=mask.device), mask], 1)).last_hidden_state[:, self.n_prompt:]
        else: h = self.backbone(input_ids=ids, attention_mask=mask).last_hidden_state
        mm = mask.unsqueeze(-1).to(h.dtype); return F.normalize(((h * mm).sum(1) / mm.sum(1).clamp(min=1)).float(), dim=-1)
def pad_chunks(chs):
    L = max(len(c) for c in chs); ids = torch.full((len(chs), L), PAD, dtype=torch.long); mask = torch.zeros((len(chs), L), dtype=torch.long)
    for i, c in enumerate(chs): ids[i, :len(c)] = torch.from_numpy(c.astype(np.int64)); mask[i, :len(c)] = 1
    return ids, mask
@torch.no_grad()
def encode_docs(model, docs, bs=None):
    """docs: list of chunk lists -> (E [n_docs, max_chunks, H] float16, M [n_docs, max_chunks] bool) on the CPU"""
    model.eval(); flat = [(d, k, c) for d, chs in enumerate(docs) for k, c in enumerate(chs)]; order = sorted(range(len(flat)), key=lambda i: len(flat[i][2])); bs = bs or (64 if device.type == "cuda" else 16)
    kmax = max(len(chs) for chs in docs); E = torch.zeros(len(docs), kmax, model.hidden, dtype=torch.float16); M = torch.zeros(len(docs), kmax, dtype=torch.bool)
    for j in range(0, len(order), bs):
        part = [flat[i] for i in order[j:j + bs]]; ids, mask = pad_chunks([p[2] for p in part])
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP): v = model(ids.to(device), mask.to(device))
        v = v.half().cpu()
        for (d, k, _), x in zip(part, v): E[d, k] = x; M[d, k] = True
    return E, M
@torch.no_grad()
def score(QE, QM, CE, CM, mode):
    if mode == "mean":
        q = F.normalize((QE.float() * QM[..., None]).sum(1) / QM.sum(1, keepdim=True).clamp(min=1), dim=-1); c = F.normalize((CE.float() * CM[..., None]).sum(1) / CM.sum(1, keepdim=True).clamp(min=1), dim=-1); return (q @ c.T).numpy()
    dev_ = device; C = CE.to(dev_).float(); Cm = CM.to(dev_); out = np.zeros((QE.shape[0], CE.shape[0]), dtype=np.float32)
    for i in range(0, QE.shape[0], 8):
        q = QE[i:i + 8].to(dev_).float(); qm = QM[i:i + 8].to(dev_); s = torch.einsum("qah,cbh->qcab", q, C).masked_fill(~Cm[None, :, None, :], -1e4).amax(-1)
        out[i:i + 8] = ((s * qm[:, None, :]).sum(-1) / qm.sum(-1, keepdim=True).clamp(min=1)).cpu().numpy()
    return out
def evaluate_model(model, name, technique, note, splits=("dev", "test"), extra=None, control=False, chunk_m=None, adopted=True):
    """encode dev + test (queries and their OWN pools), score both ways, keep the better mode on dev, save everything"""
    t0 = time.time(); enc = {s: (encode_docs(model, chunks_for(s, "q", chunk_m)), encode_docs(model, chunks_for(s, "c", chunk_m))) for s in splits}; res = {}
    for mode in ("mean", "maxsim"): S = score(*enc["dev"][0], *enc["dev"][1], mode); res[mode] = (micro_at_k(rank_top(S, DATA["dev"]), DATA["dev"]["gold"]), S)
    mode = max(res, key=lambda m: res[m][0]["f1"]); m, Sd = res[mode]; St = score(*enc["test"][0], *enc["test"][1], mode); secs = time.time() - t0
    print(f"[{name}] dev micro-F1@7 {m['f1']:.4f} (P@7 {m['p']:.4f}, R@7 {m['r']:.4f}) with '{mode}' scoring | other mode: {res['mean' if mode == 'maxsim' else 'maxsim'][0]['f1']:.4f} | encoding + scoring {secs / 60:.1f} min")
    save_scores(name, technique, note, Sd, St, m, extra=dict(mode=mode, chunks=chunk_m or MAX_CHUNKS, other_mode_f1=res["mean" if mode == "maxsim" else "maxsim"][0]["f1"], eval_minutes=secs / 60, **(extra or {})), control=control, adopted=adopted); return m
def save_scores(name, technique, note, S_dev, S_test, m, extra=None, control=False, adopted=True):
    rd = WORK / "runs" / name; rd.mkdir(parents=True, exist_ok=True); np.save(rd / "dev_scores.npy", S_dev.astype(np.float16)); np.save(rd / "test_scores.npy", S_test.astype(np.float16))
    res = dict(name=name, technique=technique, note=note, control=control, adopted=bool(adopted), dev_f1=m["f1"], dev_p=m["p"], dev_r=m["r"], dev_queries=len(DATA["dev"]["q_ids"]), **(extra or {})); json.dump(res, open(rd / "result.json", "w"), indent=1, default=str)
    led = pd.read_csv(LEDGER) if LEDGER.exists() else pd.DataFrame(); led = pd.concat([led[led.name != name] if len(led) else led, pd.DataFrame([{k: v for k, v in res.items() if not isinstance(v, (dict, list))}])]); led.to_csv(LEDGER, index=False)
    update_final(verbose=False); return res
def run_done(name): return (WORK / "runs" / name / "result.json").exists() and not RERUN


In [ ]:
# ================= CELL 13 — RESTORE AN EARLIER RUN OF THIS NOTEBOOK  [optional; RUNS NOW] =================
# If the output of an earlier (interrupted) run of this notebook is attached (Add Input > Notebook Output Files), its finished runs, checkpoints and ledger are copied back and are NOT trained again.
n_rest = 0
for src in sorted({Path(p).parent for p in glob.glob("/kaggle/input/**/pcr/ledger.csv", recursive=True)} - {WORK}):
    for sub in ("runs", "checkpoints"):
        if (src / sub).exists(): shutil.copytree(src / sub, WORK / sub, dirs_exist_ok=True); n_rest += sum(1 for _ in (src / sub).rglob("*"))
    old = pd.read_csv(src / "ledger.csv"); cur = pd.read_csv(LEDGER) if LEDGER.exists() else pd.DataFrame(); pd.concat([old, cur]).drop_duplicates("name", keep="last").to_csv(LEDGER, index=False); print("restored from", src)
print(f"restored {n_rest} files" if n_rest else "nothing to restore (a fresh run)")


In [ ]:
# ================= CELL 9 — LEXICAL BASELINE (BM25) AND HARD NEGATIVES  [RUNS NOW; CPU, a few minutes] =================
# BM25 is a baseline and the miner of HARD NEGATIVES for training (top-ranked candidates that are NOT relevant). Under RULE_MODE = "strict" it never decides the final ranking; it only appears in prediction_hybrid.json.
from sklearn.feature_extraction.text import CountVectorizer
import scipy.sparse as sp
STOP = "the of and to in a is that be by for was as it on with or this not he which his are has an at from any have under were been had shall said such no but if may their they there its upon would other these those than also into made court section".split()
def bm25_matrix(D, k1=1.2, b=0.75, fitted=None):
    if fitted is None:
        cv = CountVectorizer(lowercase=True, token_pattern=r"(?u)\b[a-z0-9][a-z0-9]+\b", stop_words=STOP, dtype=np.float32, min_df=1); C = cv.fit_transform(D["c_text"]).tocsr(); Q = cv.transform(D["q_text"]).tocsr(); fitted = (C, Q)
    C, Q = fitted; N = C.shape[0]; dl = np.asarray(C.sum(1)).ravel(); avg = dl.mean() + 1e-9; dfreq = np.asarray((C > 0).sum(0)).ravel(); idf = np.log((N - dfreq + 0.5) / (dfreq + 0.5) + 1.0).astype(np.float32)
    Co = C.tocoo(); w = Co.data * (k1 + 1) / (Co.data + k1 * (1 - b + b * dl[Co.row] / avg)) * idf[Co.col]; W = sp.csr_matrix((w.astype(np.float32), (Co.row, Co.col)), shape=C.shape)
    Qw = Q.copy(); Qw.data = np.log1p(Qw.data); return np.asarray((Qw @ W.T).todense(), dtype=np.float32), fitted
t0 = time.time(); BM25, FIT = {}, {}
S0, FIT["dev"] = bm25_matrix(DATA["dev"]); grid = []
for k1 in (0.9, 1.2, 1.6):
    for b in (0.5, 0.75, 0.9):
        S, _ = bm25_matrix(DATA["dev"], k1, b, FIT["dev"]); grid.append((micro_at_k(rank_top(S, DATA["dev"]), DATA["dev"]["gold"])["f1"], k1, b))
BM_F1, BM_K1, BM_B = max(grid); print("BM25 dev grid (micro-F1@7, k1, b):", [(round(f, 4), k1, b) for f, k1, b in sorted(grid, reverse=True)[:4]])
for s in ("dev", "test", "train"): BM25[s], FIT[s] = bm25_matrix(DATA[s], BM_K1, BM_B, FIT.get(s))
m_bm = micro_at_k(rank_top(BM25["dev"], DATA["dev"]), DATA["dev"]["gold"]); print(f"BM25 (k1={BM_K1}, b={BM_B}) on dev: micro-F1@7 {m_bm['f1']:.4f} | P@7 {m_bm['p']:.4f} | R@7 {m_bm['r']:.4f} | {time.time() - t0:.0f}s")
HARD = []                                                                        # for every training query: the best-ranked NON-relevant training candidates (hard negatives)
for qi in range(len(DATA["train"]["q_ids"])):
    order = np.argsort(-BM25["train"][qi]); g = DATA["train"]["gold"][qi]; qself = DATA["train"]["c_pos"].get(DATA["train"]["q_ids"][qi]); HARD.append([int(c) for c in order[:60] if c not in g and c != qself][:40])
save_scores("B0_bm25", "bm25", "lexical baseline (not a Phase 1 model)", BM25["dev"], BM25["test"], m_bm, extra=dict(k1=BM_K1, b=BM_B), control=True)


In [ ]:
# ================= CELL 11 — CONTRASTIVE TRAINING (+ distillation)  [pure code; needs torch] =================
# Every step: B training queries; for each, one RELEVANT candidate and one BM25 HARD NEGATIVE from the training pool; every document is represented by K_TRAIN random chunks (mean of their vectors).
# Loss: InfoNCE over all 2B candidates of the batch (in-batch negatives + hard negatives), the other relevant candidates of a query masked out (they are not negatives). DISTILLATION adds the teacher's softmax over the same candidates (KL, temperature T).
TECH_MAP = {"full": "full", "partial": "partial", "lorma_plus": "lorma_plus", "ia3": "ia3", "kd": "full"}
DEFAULT_LR = {"full": 2e-5, "partial": 3e-5, "lorma_plus": 5e-4, "ia3": 3e-3, "prompt": 3e-3, "kd": 3e-5}
def build_encoder(technique, hp=None, n_layers=None, n_prompt=20):
    bb, hidden, causal, ctx, fam = load_backbone_any(MODEL_PATHS["modernbert"]); assert fam == "modernbert"
    if n_layers: truncate_modernbert(bb, n_layers)
    model = DocEncoder(bb, hidden, n_prompt if technique == "prompt" else 0)
    if technique == "prompt":
        for p in model.backbone.parameters(): p.requires_grad = False
        model.prompt.requires_grad = True
    else: apply_method(model, TECH_MAP[technique], hp or {})
    return model
def trainable_state(model): names = {n for n, p in model.named_parameters() if p.requires_grad}; return {k: v.detach().cpu() for k, v in model.state_dict().items() if k in names}
def load_retriever(name):
    """a scores-only VARIANT of a run (for example '<run>_c24', scored with more chunks) has no checkpoint of its own: it resolves to its base run's checkpoint and adapter settings; the meta returned is the requested run's own (its scoring mode, its dev score)"""
    meta = json.load(open(WORK / "runs" / name / "result.json")); base = meta.get("base_run", name); bmeta = json.load(open(WORK / "runs" / base / "result.json")) if base != name else meta
    model = build_encoder(bmeta["technique"], bmeta.get("hp"), bmeta.get("n_layers"), bmeta.get("n_prompt", 20)); sd = torch.load(CKPT / f"{base}.pt", map_location="cpu", weights_only=False)
    miss = model.load_state_dict(sd, strict=False); assert not miss.unexpected_keys, miss.unexpected_keys[:3]; return model.to(device), meta
def base_meta(name): m = json.load(open(WORK / "runs" / name / "result.json")); return json.load(open(WORK / "runs" / m.get("base_run", name) / "result.json")) if m.get("base_run", name) != name else m
def train_retriever(name, technique, hp=None, minutes=30, lr=None, n_layers=None, teacher=None, kd_alpha=0.5, kd_T=2.0, B=8, K_TRAIN=2, tau=0.05, n_prompt=20, seed=SEED, note="", hard=None, n_neg=1, control=False):
    if run_done(name) and (CKPT / f"{name}.pt").exists(): r = json.load(open(WORK / "runs" / name / "result.json")); print(f"[{name}] already finished (restored): dev micro-F1@7 {r['dev_f1']:.4f}"); return r
    seed_all(seed); rng = np.random.default_rng(seed); t0 = time.time(); model = build_encoder(technique, hp, n_layers, n_prompt).to(device); lr = lr or DEFAULT_LR[technique]
    if technique in ("full", "partial", "kd") and device.type == "cuda" and hasattr(model.backbone.inner, "gradient_checkpointing_enable"): model.backbone.inner.gradient_checkpointing_enable()
    params = [p for p in model.parameters() if p.requires_grad]; n_tr, n_all = sum(p.numel() for p in params), sum(p.numel() for p in model.parameters())
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.01); scaler = torch.cuda.amp.GradScaler(enabled=AMP); Dt = DATA["train"]; TRQ = [q for q in range(len(Dt["q_ids"])) if Dt["gold"][q]]
    print(f"[{name}] {technique} | {n_all / 1e6:.1f}M params, {n_tr / 1e6:.2f}M trained ({100 * n_tr / n_all:.2f}%) | lr {lr} | {len(TRQ)} training queries | B={B}, {K_TRAIN} chunks per document | cap {minutes} min" + (f" | teacher {teacher[1]}" if teacher else ""))
    def pick(chs): return [chs[i] for i in rng.choice(len(chs), size=min(K_TRAIN, len(chs)), replace=False)]
    def doc_vecs(m, docs):
        flat = [c for d in docs for c in d]; ids, mask = pad_chunks(flat)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP): v = m(ids.to(device), mask.to(device))
        out, j = [], 0
        for d in docs: out.append(F.normalize(v[j:j + len(d)].float().mean(0), dim=-1)); j += len(d)
        return torch.stack(out)
    est_steps, step, losses, warm = None, 0, [], 30; model.train()
    while True:
        el = (time.time() - t0) / 60
        if el >= minutes * 0.95: break
        qs = rng.choice(TRQ, size=min(B, len(TRQ)), replace=False); pos = [int(rng.choice(sorted(Dt["gold"][q]))) for q in qs]; HH = hard if hard is not None else HARD; neg = [int(x) for q in qs for x in (rng.choice(HH[q], size=min(n_neg, len(HH[q])), replace=False) if HH[q] else rng.integers(0, len(Dt["c_ids"]), size=n_neg))]
        cands = pos + neg; qd = [pick(Dt["q_chunks"][q]) for q in qs]; cd = [pick(Dt["c_chunks"][c]) for c in cands]
        mask_ = torch.zeros(len(qs), len(cands), dtype=torch.bool)
        for i, q in enumerate(qs):
            for j, c in enumerate(cands):
                if j != i and c in Dt["gold"][q]: mask_[i, j] = True                # another relevant candidate of the same query is not a negative
        qv, cv = doc_vecs(model, qd), doc_vecs(model, cd); logits = (qv @ cv.T / tau).masked_fill(mask_.to(device), -1e4); tgt = torch.arange(len(qs), device=device); loss = F.cross_entropy(logits, tgt)
        if teacher is not None:
            with torch.no_grad(): tl = (doc_vecs(teacher[0], qd) @ doc_vecs(teacher[0], cd).T / tau).masked_fill(mask_.to(device), -1e4)
            loss = kd_alpha * loss + (1 - kd_alpha) * kd_T ** 2 * F.kl_div(F.log_softmax(logits / kd_T, -1), F.softmax(tl / kd_T, -1), reduction="batchmean")
        lr_now = lr * min(1.0, (step + 1) / warm); [g.__setitem__("lr", lr_now) for g in opt.param_groups]
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0); scaler.step(opt); scaler.update(); step += 1; losses.append(float(loss))
        if step % 50 == 0: print(f"   step {step} | loss {np.mean(losses[-50:]):.4f} | {el:.1f} min", flush=True)
    train_min = (time.time() - t0) / 60; print(f"   trained {step} steps in {train_min:.1f} min (last-50 loss {np.mean(losses[-50:]) if losses else float('nan'):.4f})")
    if hasattr(model.backbone.inner, "gradient_checkpointing_disable"): model.backbone.inner.gradient_checkpointing_disable()
    torch.save(trainable_state(model), CKPT / f"{name}.pt"); mem = torch.cuda.max_memory_allocated() / 2**30 if device.type == "cuda" else float("nan")
    t1 = time.time(); m = evaluate_model(model, name, technique, note, extra=dict(hp=hp, n_layers=n_layers, n_prompt=n_prompt if technique == "prompt" else None, lr=lr, steps=step, train_minutes=train_min, params_M=n_all / 1e6, trainable_M=n_tr / 1e6, peak_mem_gb=mem, teacher=teacher[1] if teacher else None, n_neg=n_neg, hard_negatives="round 2 (mined by a trained model)" if hard is not None else "BM25"), control=control)
    r = json.load(open(WORK / "runs" / name / "result.json")); del model, opt; gc.collect()
    if device.type == "cuda": torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    return r


In [ ]:
# ================= CELL 14 — SETTINGS, TIME BUDGET AND HELPERS  [pure code] =================
# PART 2 of the Phase 7 notebook: restores every finished run of Part 1 (nothing is retrained) and adds, in order of value / cheapness, the techniques that can move the score most. Every stage checks the time budget first and SKIPS itself if it would not fit;
# every stage is ADOPTED only if it passes a paired-bootstrap rule on dev (a stage that fails is kept in the table, flagged, and never becomes prediction.json). prediction.json is rewritten after every stage.
TOTAL_BUDGET_MIN, FINAL_RESERVE_MIN = (420, 15) if TIME_PROFILE == "normal" else (240, 10)
CAPS2 = dict(R2=35, XENC=25) if TIME_PROFILE == "normal" else dict(R2=18, XENC=12)
RUN_STAGES = {"POOL", "NGRAM", "ENS", "SWEEP", "R2", "CTX", "XENC"}                         # remove a name to skip a stage
MIN_GAIN = 0.005                                                                            # a stage is adopted if dev micro-F1@7 rises by at least this AND the paired 95% interval of the gain is above 0
if SELFTEST: TOTAL_BUDGET_MIN, CAPS2, MIN_GAIN = 999, {k: 0.15 for k in CAPS2}, 0.0
def left_min(): return TOTAL_BUDGET_MIN - (time.time() - NB_T0) / 60
def can_run(label, need):
    ok = left_min() - FINAL_RESERVE_MIN >= need
    if not ok: print(f"SKIPPED {label}: needs about {need:.0f} min but only {left_min() - FINAL_RESERVE_MIN:.0f} min of the {TOTAL_BUDGET_MIN}-minute budget remain")
    return ok
def z(S): S = S.astype(np.float32); return (S - S.mean(1, keepdims=True)) / (S.std(1, keepdims=True) + 1e-6)
def load_sc(n, s): return np.load(WORK / "runs" / n / f"{s}_scores.npy").astype(np.float32)
def f1_of(S, s="dev"): return micro_at_k(rank_top(S, DATA[s]), DATA[s]["gold"])
def gain(S_new, S_ref):
    m_new, m_ref = f1_of(S_new), f1_of(S_ref); d, lo, hi = boot_diff(m_new, m_ref); return m_new["f1"] - m_ref["f1"], lo, hi, m_new
def passes(S_new, S_ref, min_gain=None, lb=0.0): d, lo, hi, _ = gain(S_new, S_ref); return (d >= (MIN_GAIN if min_gain is None else min_gain) and lo > lb), d, lo, hi
def neural_names():
    T = runs_table(); T = T[(T.technique.isin(["full", "partial", "lorma_plus", "ia3", "prompt", "kd"])) & (~T.control.astype(bool))] if len(T) else T; return list(T.sort_values("dev_f1", ascending=False).name) if len(T) else []
NEURAL = neural_names(); print(f"budget {TOTAL_BUDGET_MIN} min ({(time.time() - NB_T0) / 60:.0f} used) | stages {sorted(RUN_STAGES)} | restored neural runs: {NEURAL}"); assert NEURAL, "no finished neural run was restored: attach the output of Part 1 (Add Input > Notebook Output Files)"


In [ ]:
# ================= CELL 15 — THE STRUCTURE OF THE POOLS AND OF THE TEXT  [RUNS NOW; CPU; seconds] =================
# (a) HOW THE POOLS ARE BUILT. If nearly every pool document is relevant to about one query of the same split, queries COMPETE for candidates, and a candidate that fits another query better is probably not relevant here (pool-aware re-ranking, next cell).
# (b) CITATION MARKERS. If the query texts keep a placeholder where a citation was removed, the sentences around it are the ideal retrieval query (stage CTX).
rows = []
for s, D in DATA.items():
    n_q, n_c = len(D["q_ids"]), len(D["c_ids"]); qset = set(D["q_ids"]); in_pool = sum(q in D["c_pos"] for q in D["q_ids"]); row = dict(split=s, queries=n_q, pool=n_c, queries_in_pool=in_pool, pool_per_query=round(n_c / n_q, 2))
    if D["gold"]:
        cited = [set() for _ in range(n_c)]
        for qi, g in enumerate(D["gold"]):
            for c in g: cited[c].add(qi)
        n_cited = sum(1 for x in cited if x); pairs = sum(len(g) for g in D["gold"]); row.update(gold_pairs=pairs, pool_docs_relevant_to_a_query=f"{n_cited / n_c:.0%}", queries_per_cited_doc=round(pairs / max(1, n_cited), 2), non_query_docs_relevant=f"{sum(1 for c, x in enumerate(cited) if x and D['c_ids'][c] not in qset) / max(1, n_c - in_pool):.0%}")
    rows.append(row)
print("POOL STRUCTURE:"); print(pd.DataFrame(rows).to_string(index=False))
MARK_PATTERNS = [r"<[A-Za-z_ ]{2,30}>", r"\[[A-Za-z_ ]{2,30}\]", r"\(\s*(?:citation|cit)\s*\)", r"X{3,}", r"\*{3,}", r"_{3,}"]; MARKER = None; best = (0.0, None)
for pat in MARK_PATTERNS:
    rx = re.compile(pat, re.I if "cit" in pat else 0); allm = [m for t in DATA["train"]["q_text"][:400] for m in rx.findall(t)]
    if not allm: continue
    top, cnt = Counter(allm).most_common(1)[0]; share_q = float(np.mean([bool(rx.search(t)) for t in DATA["train"]["q_text"][:400]])); share_c = float(np.mean([bool(rx.search(t)) for t in DATA["train"]["c_text"][:400]]))
    print(f"  marker pattern {pat!r}: most common match {top!r} ({cnt / len(allm):.0%} of {len(allm)} matches) | in {share_q:.0%} of queries, {share_c:.0%} of candidates")
    if cnt / len(allm) >= 0.7 and share_q >= 0.5 and share_q > best[0]: best = (share_q, (pat, top))
if best[1]: MARKER = best[1]; print(f"=> a placeholder looks present: {MARKER[1]!r} (pattern {MARKER[0]!r}): the CTX stage will build citation-context queries")
else: print("=> no repeated placeholder found in the query texts: the CTX stage will skip itself")
print("a query text starts:", repr(DATA["train"]["q_text"][0][:240]))


In [ ]:
# ================= CELL 16 — UNTRAINED SCORES ON THE TRAINING SPLIT  [BM25: CPU, seconds; zero-shot Phase 1 encoder: needs a GPU, about 6-10 min] =================
# Pool-aware re-ranking has 2-3 parameters. To tune them WITHOUT touching the dev labels, they are tuned on the TRAINING split (827 queries) with scorers that never saw training labels: BM25 and the untouched Phase 1 encoder. Dev then only confirms.
TRAIN_SC = {"bm25": BM25["train"]}
if "POOL" in RUN_STAGES and "modernbert" in MODEL_PATHS and can_run("zero-shot scores of the training split", 12):
    def z0train():
        bb, hidden, *_ = load_backbone_any(MODEL_PATHS["modernbert"]); m = DocEncoder(bb, hidden).to(device); t0 = time.time(); Q, C = encode_docs(m, chunks_for("train", "q")), encode_docs(m, chunks_for("train", "c"))
        mode = json.load(open(WORK / "runs" / "Z0_phase1_zero_shot" / "result.json")).get("mode", "maxsim") if (WORK / "runs" / "Z0_phase1_zero_shot" / "result.json").exists() else "maxsim"; TRAIN_SC["z0"] = score(*Q, *C, mode); print(f"zero-shot Phase 1 scores of the training split ({mode}): micro-F1@7 {f1_of(TRAIN_SC['z0'], 'train')['f1']:.4f} | {(time.time() - t0) / 60:.1f} min"); del m; gc.collect()
        if device.type == "cuda": torch.cuda.empty_cache()
    safe("zero-shot train scores", z0train)
print("BM25 on the training split: micro-F1@7 %.4f" % f1_of(TRAIN_SC["bm25"], "train")["f1"])


In [ ]:
# ================= CELL 17 — POOL-AWARE RE-RANKING  [RUNS NOW; CPU; seconds-minutes] =================
# Scores are z-normalised per query, then a candidate-specific COMPETITION term is subtracted: the more strongly OTHER queries of the same split match a candidate, the less it is believed to belong to this one.
#   'lse'   S' = Z - beta * tau * logsumexp_over_queries(Z / tau)        (a soft version of 'subtract the best competing query'; tau -> 0 is the max)
#   'csls'  S' = Z - beta * mean of the top-k scores of the candidate over all queries   (hubness reduction)
#   'sink'  Sinkhorn normalisation: alternate row and column normalisation of exp(Z / tau)
#   'rrank' reciprocal rank: -(log(1 + rank of the candidate for the query) + lambda * log(1 + rank of the query for the candidate)); scale-free, it encodes 'a candidate belongs to about one query' directly
# The query's own document (if it sits in its own pool) is never a candidate. Only unlabeled scores of the split's own queries are used; no label and no other split.
def zrows(S, D):
    S = np.array(S, dtype=np.float32, copy=True)
    for qi, q in enumerate(D["q_ids"]):
        if q in D["c_pos"]: S[qi, D["c_pos"][q]] = np.nan
    Z = (S - np.nanmean(S, 1, keepdims=True)) / (np.nanstd(S, 1, keepdims=True) + 1e-6); return np.nan_to_num(Z, nan=-1e4)
def pool_rerank(S, D, mode, p):
    Z = zrows(S, D)
    if mode == "lse": tau, beta = p; Zt = Z / tau; m = Zt.max(0, keepdims=True); L = tau * (m[0] + np.log(np.exp(Zt - m).sum(0))); return Z - beta * L[None, :]
    if mode == "csls": k, beta = p; top = (-np.sort(-Z, axis=0)[:k]).mean(0); return Z - beta * top[None, :]
    if mode == "rrank":                                                              # RECIPROCAL RANK: how highly this candidate ranks for the query AND how highly this query ranks among all queries for the candidate (a candidate belongs to ~one query)
        lam, _ = p; R = np.argsort(np.argsort(-Z, axis=1), axis=1); C = np.argsort(np.argsort(-Z, axis=0), axis=0); return -(np.log1p(R) + lam * np.log1p(C)) + 1e-3 * Z
    if mode == "sink":
        tau, it = p; K = np.exp(np.clip(Z / tau, -50, 50)); 
        for _ in range(it): K = K / (K.sum(1, keepdims=True) + 1e-12); K = K / (K.sum(0, keepdims=True) + 1e-12)
        return np.log(K + 1e-30)
    raise ValueError(mode)
POOL_GRID = [("lse", (t, b)) for t in (0.1, 0.2, 0.5, 1.0) for b in (0.25, 0.5, 0.75, 0.9)] + [("csls", (k, b)) for k in (3, 10) for b in (0.5, 1.0)] + [("sink", (t, 3)) for t in (0.2, 0.5)] + [("rrank", (l, 0)) for l in (0.5, 1.0, 2.0)]
POOL_PARAMS, POOL_REPORT = None, []
if "POOL" in RUN_STAGES:
    def f1_split(S, s): return f1_of(pool_rerank(S, DATA[s], *g), s)["f1"]
    base = {k: f1_of(zrows(v, DATA["train"]), "train")["f1"] for k, v in TRAIN_SC.items()}; res = []
    for g in POOL_GRID:
        gains = {k: f1_of(pool_rerank(v, DATA["train"], *g), "train")["f1"] - base[k] for k, v in TRAIN_SC.items()}; res.append((float(np.mean(list(gains.values()))), g, gains))
    res.sort(key=lambda r: -r[0]); POOL_PARAMS = res[0][1]
    print("TRAINING SPLIT (untrained scorers; baseline micro-F1@7:", {k: round(v, 4) for k, v in base.items()}, ") best re-ranking settings, gain = micro-F1@7 after - before:")
    for mg, g, gains in res[:5]: print(f"   {g[0]:5s} {g[1]}  mean gain {mg:+.4f}  " + "  ".join(f"{k} {v:+.4f}" for k, v in gains.items()))
    print(f"=> settings chosen on the TRAINING split (no dev label touched): {POOL_PARAMS}")
    base_name = NEURAL[0]; ENS_NAMES = NEURAL[:2]; S_ref = (z(load_sc(ENS_NAMES[0], "dev")) + z(load_sc(ENS_NAMES[-1], "dev"))) / 2 if len(ENS_NAMES) > 1 else z(load_sc(base_name, "dev"))
    rows = []
    for label, S in [("BM25 (untrained)", BM25["dev"]), ("zero-shot Phase 1", load_sc("Z0_phase1_zero_shot", "dev")), (f"neural ensemble {ENS_NAMES}", S_ref)]:
        ok, d, lo, hi = passes(pool_rerank(S, DATA["dev"], *POOL_PARAMS), zrows(S, DATA["dev"]), min_gain=-1.0, lb=-9.0); rows.append(dict(scores=label, dev_before=round(f1_of(zrows(S, DATA["dev"]))["f1"], 4), dev_after=round(f1_of(pool_rerank(S, DATA["dev"], *POOL_PARAMS))["f1"], 4), gain=f"{d:+.4f} [{lo:+.4f}, {hi:+.4f}]"))
    POOL_REPORT = pd.DataFrame(rows); print("\nCONFIRMATION ON DEV (settings fixed on the training split):"); print(POOL_REPORT.to_string(index=False))
    ok, d, lo, hi = passes(pool_rerank(S_ref, DATA["dev"], *POOL_PARAMS), zrows(S_ref, DATA["dev"])); POOL_ADOPTED = bool(ok)
    if not ok:                                                                           # fallback: tune on dev (2 parameters) with a stricter bar
        devres = sorted(((f1_of(pool_rerank(S_ref, DATA["dev"], *g))["f1"], g) for g in POOL_GRID), key=lambda r: -r[0]); g2 = devres[0][1]; ok2, d2, lo2, hi2 = passes(pool_rerank(S_ref, DATA["dev"], *g2), zrows(S_ref, DATA["dev"]), min_gain=0.01, lb=0.0)
        print(f"train-tuned settings not adopted ({d:+.4f} [{lo:+.4f}, {hi:+.4f}]); dev-tuned {g2}: {d2:+.4f} [{lo2:+.4f}, {hi2:+.4f}] -> {'ADOPTED (stricter bar: +0.01 and interval above 0)' if ok2 else 'not adopted'}"); POOL_ADOPTED, POOL_PARAMS = bool(ok2), (g2 if ok2 else POOL_PARAMS)
    else: print(f"=> pool-aware re-ranking ADOPTED: {d:+.4f} [{lo:+.4f}, {hi:+.4f}] on dev with settings chosen on the training split")
    Sd, St = pool_rerank(S_ref, DATA["dev"], *POOL_PARAMS), pool_rerank((z(load_sc(ENS_NAMES[0], "test")) + z(load_sc(ENS_NAMES[-1], "test"))) / 2 if len(ENS_NAMES) > 1 else z(load_sc(base_name, "test")), DATA["test"], *POOL_PARAMS)
    save_scores("C2_pool_early", "pool", f"pool-aware re-ranking {POOL_PARAMS} of the mean of {ENS_NAMES}", Sd, St, f1_of(Sd), extra=dict(params=str(POOL_PARAMS), members=ENS_NAMES), control=True, adopted=POOL_ADOPTED); update_final()


In [ ]:
# ================= CELL 18 — N-GRAM BM25 FOR THE HYBRID TRACK  [RUNS NOW; CPU; about 5-15 min] =================
# On this very benchmark (IL-PCR) published work finds BM25 variants on word n-grams strong, far above plain unigram BM25 (the paper's event-filtered quad-gram BM25 needs an outside parser, which the rules forbid; plain n-grams need nothing). Lexical = NOT a Phase 1 model:
# the result goes to the HYBRID track only. The fusion weights of unigram + 2-/3-/4-gram BM25 are tuned on the TRAINING split (no neural model involved), dev only confirms.
from sklearn.feature_extraction.text import HashingVectorizer
import itertools
def bm25_ng(D, n):
    hv = HashingVectorizer(analyzer="word", ngram_range=(n, n), n_features=2 ** 22, alternate_sign=False, norm=None, lowercase=True, token_pattern=r"(?u)\b[a-z0-9][a-z0-9]+\b", stop_words=STOP, dtype=np.float32)
    return bm25_matrix(D, BM_K1, BM_B, (hv.transform(D["c_text"]).tocsr(), hv.transform(D["q_text"]).tocsr()))[0]
NG, LEX, W_LEX = {}, None, None
if "NGRAM" in RUN_STAGES and can_run("n-gram BM25", 15):
    def ng():
        global LEX, W_LEX
        for n in (2, 3, 4):
            NG[n] = {s: bm25_ng(DATA[s], n) for s in ("train", "dev", "test")}; print(f"  BM25 on {n}-grams: train micro-F1@7 {f1_of(NG[n]['train'], 'train')['f1']:.4f} | dev {f1_of(NG[n]['dev'])['f1']:.4f}", flush=True)
        best = None
        for w in itertools.product((0, 0.5, 1), repeat=3):
            S = z(BM25["train"]) + sum(wi * z(NG[n]["train"]) for wi, n in zip(w, (2, 3, 4)) if wi); f = f1_of(S, "train")["f1"]
            if best is None or f > best[0]: best = (f, w)
        f_tr, W_LEX = best; LEX = {s: z(BM25[s]) + sum(wi * z(NG[n][s]) for wi, n in zip(W_LEX, (2, 3, 4)) if wi) for s in ("dev", "test")}; m = f1_of(LEX["dev"])
        print(f"lexical fusion, weights tuned on the TRAINING split (unigram + {dict(zip((2, 3, 4), W_LEX))}): train {f_tr:.4f} | dev micro-F1@7 {m['f1']:.4f} (plain unigram BM25: {f1_of(BM25['dev'])['f1']:.4f})")
        save_scores("B1_bm25_ngrams", "bm25", f"lexical: unigram + n-gram BM25 fusion {W_LEX}, weights tuned on the training split (not a Phase 1 model)", LEX["dev"], LEX["test"], m, extra=dict(weights=str(W_LEX)), control=True)
    safe("n-gram BM25", ng)


In [ ]:
# ================= CELL 19 — ENSEMBLE OF ALL RUNS AND THE CHUNK SWEEP  [ensemble: seconds; sweep needs a GPU, about 5-10 min per model] =================
# (1) ENSEMBLE: the five finished techniques score the same documents in different ways; averaging their per-query standardised scores is free. The LARGEST ensemble within 0.003 of the best dev score is used (robust rather than chasing noise).
# (2) CHUNK SWEEP: 47% of the documents are longer than 12 chunks. The best two models score dev with MAX_CHUNKS_BIG chunks per document; adopted per model only if dev micro-F1@7 rises by MIN_GAIN with a paired interval above 0 (then test is scored too).
EVAL_MIN2 = float(json.load(open(WORK / "runs" / "Z0_phase1_zero_shot" / "result.json")).get("eval_minutes", 4.0)) if (WORK / "runs" / "Z0_phase1_zero_shot" / "result.json").exists() else 4.0
NEURAL_TECH = ["full", "partial", "lorma_plus", "ia3", "prompt", "kd"]
def ok_adopt(a): return not (a is False or str(a) == "False")
def pool_names():
    """every finished neural system that is adopted; of two variants of the same model (e.g. X and X_c24) only the better one"""
    T = runs_table(); T = T[T.technique.isin(NEURAL_TECH) & T.adopted.map(ok_adopt)] if len(T) else T; T = T.assign(base=T.name.str.replace("_c%d" % MAX_CHUNKS_BIG, "", regex=False)).sort_values("dev_f1", ascending=False).drop_duplicates("base"); return list(T.name)
def ens_scores(names, s): return sum(z(load_sc(n, s)) for n in names) / len(names)
def build_ens(tag="N_ens"):
    names = pool_names(); sizes = sorted({k for k in (2, 3, 5, len(names)) if 1 <= k <= len(names)}); res = {k: f1_of(ens_scores(names[:k], "dev"))["f1"] for k in sizes}; bestf = max(res.values()); k = max(kk for kk, f in res.items() if f >= bestf - 0.003)
    Sd, St = ens_scores(names[:k], "dev"), ens_scores(names[:k], "test"); print(f"ensemble sizes on dev {({kk: round(f, 4) for kk, f in res.items()})} -> the largest within 0.003 of the best: top {k} of {names}"); return names[:k], Sd, St
if "ENS" in RUN_STAGES:
    names, Sd, St = build_ens(); save_scores(f"N_ens_all_top{len(names)}", "ensemble", f"mean of standardised scores of {names}", Sd, St, f1_of(Sd), extra=dict(members=names), control=True)
if "SWEEP" in RUN_STAGES:
    for name in neural_names()[:2]:                                                          # the base runs (a variant has no checkpoint of its own)
        if not can_run(f"chunk sweep of {name}", 2.2 * EVAL_MIN2 + 2): continue
        def sweep(name=name):
            model, meta = load_retriever(name); mode = meta.get("mode", "maxsim"); Q, C = encode_docs(model, chunks_for("dev", "q", MAX_CHUNKS_BIG)), encode_docs(model, chunks_for("dev", "c", MAX_CHUNKS_BIG)); Sd = score(*Q, *C, mode); ok, d, lo, hi = passes(Sd, load_sc(name, "dev"))
            print(f"  {name}: {MAX_CHUNKS} -> {MAX_CHUNKS_BIG} chunks per document: dev micro-F1@7 {f1_of(load_sc(name, 'dev'))['f1']:.4f} -> {f1_of(Sd)['f1']:.4f} ({d:+.4f} [{lo:+.4f}, {hi:+.4f}]) -> {'ADOPTED' if ok else 'not adopted'}", flush=True)
            if ok:
                Qt, Ct = encode_docs(model, chunks_for("test", "q", MAX_CHUNKS_BIG)), encode_docs(model, chunks_for("test", "c", MAX_CHUNKS_BIG)); St = score(*Qt, *Ct, mode)
                save_scores(f"{name}_c{MAX_CHUNKS_BIG}", meta["technique"], f"{name} scored with {MAX_CHUNKS_BIG} chunks per document", Sd, St, f1_of(Sd), extra=dict(mode=mode, chunks=MAX_CHUNKS_BIG, base_run=name, trainable_M=meta.get("trainable_M"), params_M=meta.get("params_M")), control=True)
            del model; gc.collect()
            if device.type == "cuda": torch.cuda.empty_cache()
        safe(f"sweep {name}", sweep)
    if "ENS" in RUN_STAGES: names, Sd, St = build_ens(); save_scores(f"N_ens_all_top{len(names)}", "ensemble", f"mean of standardised scores of {names}", Sd, St, f1_of(Sd), extra=dict(members=names), control=True)


In [ ]:
# ================= CELL 20 — ROUND-2 HARD NEGATIVES  [NEEDS the ModernBERT Phase 1 checkpoint + a GPU; mining about 6-10 min, each run about CAPS2 + 6 min] =================
# Round 1 mined negatives with BM25. Now the best TRAINED model ranks the training pool and its top-ranked non-relevant documents (skipping the first HN_SKIP: likeliest unlabeled positives) become the hard negatives; every query sees n_neg = 3 of them per step (8 queries x (1 + 3) documents:
# many more negatives per query than before). The two best techniques of Part 1 are trained again FROM THE PHASE 1 WEIGHTS with them (variants, not counted as new experiments); they join the ensemble if they are good.
HN_SKIP, HARD2, TR_ENC = 3, None, None
if "R2" in RUN_STAGES and neural_names():
    t_name = next((n for n in pool_names() if json.load(open(WORK / "runs" / n / "result.json"))["technique"] != "kd"), None)
    if t_name and can_run("hard-negative mining", 1.8 * EVAL_MIN2 + 3):
        def mine():
            global HARD2, TR_ENC
            model, meta = load_retriever(t_name); mode = meta.get("mode", "maxsim"); Dt = DATA["train"]; Q, C = encode_docs(model, chunks_for("train", "q")), encode_docs(model, chunks_for("train", "c")); TR_ENC = (Q, C); S = score(*Q, *C, mode); HARD2 = []
            for qi in range(len(Dt["q_ids"])):
                order = np.argsort(-S[qi]); self_c = Dt["c_pos"].get(Dt["q_ids"][qi]); cand = [int(c) for c in order[:150] if c not in Dt["gold"][qi] and c != self_c]; HARD2.append(cand[HN_SKIP:HN_SKIP + 40])
            print(f"round-2 hard negatives mined with {t_name} ({mode}); train micro-F1@7 of that model (it trained on these queries): {f1_of(S, 'train')['f1']:.4f}"); del model; gc.collect()
            if device.type == "cuda": torch.cuda.empty_cache()
        safe("mining", mine)
    if HARD2 is not None:
        seen = set()
        for n in pool_names():
            r = base_meta(n); tech = r["technique"]
            if tech == "kd" or tech in seen or len(seen) >= 2: continue
            seen.add(tech); nm = f"R2_{tech}_hn"
            if can_run(nm, CAPS2["R2"] + EVAL_MIN2 + 2): safe(nm, lambda: train_retriever(nm, tech, r.get("hp"), minutes=CAPS2["R2"], hard=HARD2, n_neg=3, note=f"{tech} with round-2 hard negatives (3 per query, mined by {t_name})", control=True))
    update_final()


In [ ]:
# ================= CELL 21 — CITATION-CONTEXT QUERIES  [only if a placeholder was found in the data; NEEDS a GPU; about 2 x one evaluation per model] =================
# The sentences around the placeholder (+-1,500 characters, windows merged) describe what the missing precedent was cited FOR: the ideal query. Scored by the best two models; the context scores are fused with the ensemble in the final assembly (weight tuned on dev, adopted only by the usual rule).
CTX_S, CTX_W = None, None
if "CTX" in RUN_STAGES and MARKER and can_run("citation-context queries", 2 * (1.2 * EVAL_MIN2) + 3):
    def ctx():
        global CTX_S
        rx = re.compile(re.escape(MARKER[1]))
        def ctx_text(t, W=1500):
            spans = [(max(0, m.start() - W), min(len(t), m.end() + W)) for m in rx.finditer(t)]
            if not spans: return t[:2 * W]
            mg = [list(spans[0])]
            for a, b in spans[1:]:
                if a <= mg[-1][1]: mg[-1][1] = max(mg[-1][1], b)
                else: mg.append([a, b])
            return " ".join(t[a:b] for a, b in mg)[:60000]
        QC = {s: [chunk_doc(e.ids) for e in tok.encode_batch([ctx_text(t) for t in DATA[s]["q_text"]], add_special_tokens=False)] for s in ("dev", "test")}; per = {s: [] for s in ("dev", "test")}
        for name in pool_names()[:2]:
            model, meta = load_retriever(name); mode = meta.get("mode", "maxsim")
            for s in ("dev", "test"): per[s].append(score(*encode_docs(model, QC[s]), *encode_docs(model, chunks_for(s, "c")), mode))
            print(f"  {name}: citation-context queries, dev micro-F1@7 {f1_of(per['dev'][-1])['f1']:.4f} (full queries: {f1_of(load_sc(name, 'dev'))['f1']:.4f})", flush=True); del model; gc.collect()
            if device.type == "cuda": torch.cuda.empty_cache()
        CTX_S = {s: sum(z(x) for x in per[s]) / len(per[s]) for s in per}; print(f"citation-context ensemble (2 models): dev micro-F1@7 {f1_of(CTX_S['dev'])['f1']:.4f}")
        save_scores("C0_ctx_queries", "ctx", "queries = the sentences around the citation placeholder, best two models", CTX_S["dev"], CTX_S["test"], f1_of(CTX_S["dev"]), control=True, adopted=False)
    safe("citation context", ctx)


In [ ]:
# ================= CELL 22 — CROSS-ENCODER RE-RANKER (training)  [NEEDS the ModernBERT Phase 1 checkpoint + a GPU; about CAPS2 XENC + 4 min] =================
# A second-stage model that READS a query chunk and a candidate chunk TOGETHER (the bi-encoder reads them apart). For every (query, candidate) the chunk pair with the best bi-encoder similarity is used (training: the best pair; inference: the best two, averaged). Trained from the Phase 1 weights with a listwise
# loss over 1 relevant + 3 round-2 hard negatives per query. It only re-orders the top-30 of the earlier stages (final assembly); adopted only by the usual rule. Only 827 training queries: it may well not help, which is why it is guarded.
XMODEL, XENC_ENC, HALF = None, None, (CHUNK_TOK - 2) // 2
class CrossEnc(nn.Module):
    def __init__(self, bb, hidden): super().__init__(); self.backbone = bb; self.head = nn.Sequential(nn.Dropout(0.1), nn.Linear(hidden, 1))
    def forward(self, ids, mask):
        T = int(mask.sum(1).max()); ids, mask = ids[:, :T], mask[:, :T]; h = self.backbone(input_ids=ids, attention_mask=mask).last_hidden_state; mm = mask.unsqueeze(-1).to(h.dtype); return self.head(((h * mm).sum(1) / mm.sum(1).clamp(min=1)).float()).squeeze(-1)
def pair_ids(qc, cc):
    a, b = list(qc[1:-1][:HALF]) if BOS is not None else list(qc[:HALF]), list(cc[1:-1][:HALF]) if BOS is not None else list(cc[:HALF])
    return np.asarray(([BOS] if BOS is not None else []) + a + ([EOS] if EOS is not None else []) + b + ([EOS] if EOS is not None else []), dtype=np.int32)
def best_pairs(enc, qi, ci, top=1):
    (QE, QM), (CE, CM) = enc; sims = QE[qi].float() @ CE[ci].float().T; sims = sims.masked_fill(~QM[qi][:, None], -9.0).masked_fill(~CM[ci][None, :], -9.0); idx = sims.flatten().topk(min(top, sims.numel())).indices
    return [(int(i) // sims.shape[1], int(i) % sims.shape[1]) for i in idx]
if "XENC" in RUN_STAGES and HARD2 is not None and TR_ENC is not None and can_run("the cross-encoder", CAPS2["XENC"] + 6):
    def trainx():
        global XMODEL, XENC_ENC
        seed_all(SEED); rng = np.random.default_rng(SEED); bb, hidden, *_ = load_backbone_any(MODEL_PATHS["modernbert"]); model = CrossEnc(bb, hidden).to(device); params = list(model.parameters()); opt = torch.optim.AdamW(params, lr=2e-5, weight_decay=0.01); scaler = torch.cuda.amp.GradScaler(enabled=AMP)
        if device.type == "cuda" and hasattr(bb.inner, "gradient_checkpointing_enable"): bb.inner.gradient_checkpointing_enable()
        Dt = DATA["train"]; TRQ = [q for q in range(len(Dt["q_ids"])) if Dt["gold"][q] and len(HARD2[q]) >= 3]; t0 = time.time(); step, losses = 0, []; model.train(); B = 4
        while (time.time() - t0) / 60 < CAPS2["XENC"] * 0.95:
            qs = rng.choice(TRQ, size=min(B, len(TRQ)), replace=False); rows = []
            for q in qs:
                pos = int(rng.choice(sorted(Dt["gold"][q]))); negs = [int(x) for x in rng.choice(HARD2[q], size=3, replace=False)]
                for c in [pos] + negs: qa, cb = best_pairs(TR_ENC, int(q), c, 1)[0]; rows.append(pair_ids(Dt["q_chunks"][q][qa], Dt["c_chunks"][c][cb]))
            ids, mask = pad_chunks(rows)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP): sc = model(ids.to(device), mask.to(device))
            loss = F.cross_entropy(sc.float().view(len(qs), 4), torch.zeros(len(qs), dtype=torch.long, device=device)); opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0); scaler.step(opt); scaler.update(); step += 1; losses.append(float(loss))
            if step % 50 == 0: print(f"   cross-encoder step {step} | loss {np.mean(losses[-50:]):.4f} | {(time.time() - t0) / 60:.1f} min", flush=True)
        print(f"cross-encoder trained {step} steps in {(time.time() - t0) / 60:.1f} min (last-50 loss {np.mean(losses[-50:]) if losses else float('nan'):.4f}; chance for 4 candidates is 1.386)")
        if hasattr(bb.inner, "gradient_checkpointing_disable"): bb.inner.gradient_checkpointing_disable()
        torch.save({k: v.detach().cpu() for k, v in model.state_dict().items()}, CKPT / "XENC_cross_encoder.pt"); model.eval(); XMODEL = model
        bi, bm = load_retriever(pool_names()[0]); XENC_ENC = {s: (encode_docs(bi, chunks_for(s, "q")), encode_docs(bi, chunks_for(s, "c"))) for s in ("dev", "test")}; del bi; gc.collect()
        if device.type == "cuda": torch.cuda.empty_cache()
    safe("cross-encoder", trainx)
@torch.no_grad()
def xenc_scores(s, S_base, K=30, top=2):
    D = DATA[s]; idx = rank_top(S_base, D, K); rows, keys = [], []
    for qi in range(len(D["q_ids"])):
        for ci in idx[qi]:
            for qa, cb in best_pairs(XENC_ENC[s], qi, int(ci), top): rows.append(pair_ids(chunks_for(s, "q")[qi][qa], chunks_for(s, "c")[int(ci)][cb])); keys.append((qi, int(ci)))
    order = sorted(range(len(rows)), key=lambda i: len(rows[i])); acc = {}
    for j in range(0, len(order), 64):
        part = order[j:j + 64]; ids, mask = pad_chunks([rows[i] for i in part])
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP): sc = XMODEL(ids.to(device), mask.to(device)).float().cpu().numpy()
        for i, v in zip(part, sc): acc.setdefault(keys[i], []).append(float(v))
    X = np.full(S_base.shape, np.nan, dtype=np.float32)
    for (qi, ci), v in acc.items(): X[qi, ci] = float(np.mean(v))
    return X
def xenc_fuse(S_base, X, alpha, D):
    Zb = zrows(S_base, D); out = Zb - 1e3
    for qi in range(Zb.shape[0]):
        idx = np.where(~np.isnan(X[qi]))[0]
        if len(idx): xz = (X[qi, idx] - X[qi, idx].mean()) / (X[qi, idx].std() + 1e-6); out[qi, idx] = Zb[qi, idx] + alpha * xz
    return out


In [ ]:
# ================= CELL 23 — FINAL ASSEMBLY  [RUNS NOW; CPU except the cross-encoder scoring; about 5-15 min] =================
# The chain, every step ADOPTED only by the dev rule (gain >= MIN_GAIN and a paired 95% interval above 0), each step applied to the output of the previous adopted one:
#   ensemble of all runs -> + citation-context fusion -> + pool-aware re-ranking -> + cross-encoder re-ranking of the top 30.   All of it uses Phase 1 models only (pool-aware re-ranking uses just their scores of the split's own unlabeled queries).
# HYBRID track (lexical n-gram BM25 enters the ranking, NOT a Phase 1 model): the chain up to the citation step + n-gram BM25 (one weight tuned on dev), then pool-aware re-ranking. Written to prediction_hybrid.json; prediction.json stays rule-safe (RULE_MODE).
names, cur_d, cur_t = build_ens(); STEPS = [dict(step="ensemble of the neural runs " + str(names), dev_f1=round(f1_of(cur_d)["f1"], 4), gain="-", adopted="base")]; pre_pool_d, pre_pool_t = cur_d, cur_t
def log_step(label, new_d, ref_d, ok, d, lo, hi): STEPS.append(dict(step=label, dev_f1=round(f1_of(new_d)["f1"], 4), gain=f"{d:+.4f} [{lo:+.4f}, {hi:+.4f}]", adopted="YES" if ok else "no"))
if CTX_S is not None:
    w = max((0.25, 0.5, 1.0, 2.0), key=lambda w_: f1_of(z(cur_d) + w_ * CTX_S["dev"])["f1"]); fd, ft = z(cur_d) + w * CTX_S["dev"], z(cur_t) + w * CTX_S["test"]; ok, d, lo, hi = passes(fd, cur_d); log_step(f"+ citation-context scores (weight {w})", fd, cur_d, ok, d, lo, hi)
    save_scores("C1_ctx_fusion", "ctx", f"ensemble + {w} x citation-context scores", fd, ft, f1_of(fd), extra=dict(w=w), control=True, adopted=ok)
    if ok: cur_d, cur_t, pre_pool_d, pre_pool_t = fd, ft, fd, ft
POOL_ON = False
if POOL_PARAMS is not None and "POOL" in RUN_STAGES:
    pd_, pt_ = pool_rerank(cur_d, DATA["dev"], *POOL_PARAMS), pool_rerank(cur_t, DATA["test"], *POOL_PARAMS); ok, d, lo, hi = passes(pd_, cur_d); log_step(f"+ pool-aware re-ranking {POOL_PARAMS} (tuned on the training split)", pd_, cur_d, ok, d, lo, hi)
    save_scores("C2_pool", "pool", f"pool-aware re-ranking {POOL_PARAMS}", pd_, pt_, f1_of(pd_), extra=dict(params=str(POOL_PARAMS)), control=True, adopted=ok)
    if ok: cur_d, cur_t, POOL_ON = pd_, pt_, True
if XMODEL is not None and XENC_ENC is not None:
    def xstep():
        global cur_d, cur_t
        Xd, Xt = xenc_scores("dev", cur_d), xenc_scores("test", cur_t); a = max((0.5, 1.0, 2.0, 5.0), key=lambda a_: f1_of(xenc_fuse(cur_d, Xd, a_, DATA["dev"]))["f1"]); fd, ft = xenc_fuse(cur_d, Xd, a, DATA["dev"]), xenc_fuse(cur_t, Xt, a, DATA["test"]); ok, d, lo, hi = passes(fd, cur_d)
        log_step(f"+ cross-encoder re-ranking of the top 30 (alpha {a})", fd, cur_d, ok, d, lo, hi); save_scores("C3_xenc", "xenc", f"cross-encoder re-ranking of the top 30, alpha {a}", fd, ft, f1_of(fd), extra=dict(alpha=a), control=True, adopted=ok)
        if ok: cur_d, cur_t = fd, ft
    safe("cross-encoder scoring", xstep)
save_scores("F_final_rule_safe", "final", "the adopted chain (Phase 1 models only)", cur_d, cur_t, f1_of(cur_d), extra=dict(steps=str([s["step"] for s in STEPS if s["adopted"] in ("YES", "base")])), control=True, adopted=True)
if LEX is not None:
    al = max(np.round(np.arange(0.0, 0.91, 0.1), 1), key=lambda a_: f1_of(z(pre_pool_d) + a_ * z(LEX["dev"]))["f1"]); hd, ht = z(pre_pool_d) + al * z(LEX["dev"]), z(pre_pool_t) + al * z(LEX["test"])
    if POOL_ON: hd, ht = pool_rerank(hd, DATA["dev"], *POOL_PARAMS), pool_rerank(ht, DATA["test"], *POOL_PARAMS)
    mh = f1_of(hd); save_scores("H_final_hybrid", "hybrid", f"Phase 1 ensemble + {al} x lexical n-gram BM25" + (" + pool-aware re-ranking" if POOL_ON else ""), hd, ht, mh, extra=dict(alpha=float(al)), control=True, adopted=True); print(f"hybrid track: neural + {al} x lexical n-gram BM25" + (" + pool-aware" if POOL_ON else "") + f": dev micro-F1@7 {mh['f1']:.4f}")
FINAL = update_final(); T = runs_table().sort_values("dev_f1", ascending=False); HYB_PATH = PRED_PATH.with_name("prediction_hybrid.json"); fin_f = float(T[T.name == FINAL].dev_f1.iloc[0]); hy = T[T.technique == "hybrid"]
if len(hy) and float(hy.iloc[0]["dev_f1"]) >= fin_f:
    write_prediction(HYB_PATH, rank_top(load_sc(hy.iloc[0]["name"], "test"), DATA["test"]), DATA["test"]); validate_prediction(HYB_PATH, DATA["test"]); zip_prediction(HYB_PATH, HYB_PATH.with_suffix(".zip")); print(f"NOTE: the best Phase 1 + lexical system ({hy.iloc[0]['name']}: dev micro-F1@7 {float(hy.iloc[0]['dev_f1']):.4f}, {float(hy.iloc[0]['dev_f1']) - fin_f:+.4f} against the rule-safe {FINAL}) is written to {HYB_PATH.name} (+ .zip); prediction.json stays Phase 1 only. Submit the hybrid only if lexical scores are allowed.")
print("\nTHE CHAIN (each step is adopted only if dev micro-F1@7 rises by >= %.3f with a paired 95%% interval above 0):" % MIN_GAIN); print(pd.DataFrame(STEPS).to_string(index=False))
ref = {r["name"]: f1_of(load_sc(r["name"], "dev")) for r in T.to_dict("records")}; rows = []
for r in T.to_dict("records"):
    d1, lo, hi = boot_diff(ref[r["name"]], ref["B0_bm25"]) if "B0_bm25" in ref and r["name"] != "B0_bm25" else (0.0, 0.0, 0.0)
    rows.append(dict(system=r["name"], technique=r["technique"], rule_safe=safe_rule(r), adopted=ok_adopt(r.get("adopted", True)), counted=not bool(r.get("control")), dev_f1=round(r["dev_f1"], 4), dev_p=round(r["dev_p"], 4), dev_r=round(r["dev_r"], 4), vs_bm25=f"{d1:+.4f} [{lo:+.4f}, {hi:+.4f}]", mode=r.get("mode")))
COMP = pd.DataFrame(rows); COMP.to_csv(WORK / "comparison.csv", index=False); print("\nCOMPARISON ON DEV (micro-F1@7; intervals: paired bootstrap over queries against unigram BM25):"); print(COMP.to_string(index=False))


In [ ]:
# ================= CELL 18 — RULE CHECK AND REPORT  [RUNS NOW] -> requirements_report.md, analysis.md =================
T = runs_table(); C = T[(~T.control.astype(bool)) & T.technique.isin(["full", "partial", "lorma_plus", "ia3", "prompt", "kd"])] if len(T) else T; fam = sorted(set(C.technique)) if len(C) else []
ok = len(C) >= 6 and len(set(fam) - {"kd"}) >= 5 and "lorma_plus" in fam and "kd" in fam; status = "ALL PHASE 7 EXPERIMENT RULES SATISFIED" if ok else "NOT YET COMPLIANT: " + "; ".join(x for x in [f"only {len(C)} counted experiments (six needed)" if len(C) < 6 else "", "no LoRMA run" if "lorma_plus" not in fam else "", "no distillation run" if "kd" not in fam else "", f"only {len(set(fam) - {'kd'})} adaptation techniques besides distillation (five needed)" if len(set(fam) - {"kd"}) < 5 else ""] if x)
fin = json.load(open(WORK / "final_choice.json")) if (WORK / "final_choice.json").exists() else {}
rep = ["# Phase 7 requirement check", "", f"**{status}**", "", f"- counted experiments: {list(C.name) if len(C) else []}", f"- techniques: {fam}", "- every experiment starts from the Phase 1 ModernBERT encoder; no outside model, weights, corpora or APIs; test labels never used",
       f"- prediction.json: {fin.get('system')} (dev micro-F1@7 {fin.get('dev_f1', float('nan')):.4f}); exactly {TOP_K} unique candidates per test query, all from the TEST pool", "", "Dev numbers are local evaluation, not official test scores."]
(WORK / "requirements_report.md").write_text("\n".join(rep), encoding="utf-8"); print("\n".join(rep))
if len(T) and {"E1_enc_full", "E6_enc_half_kd"} <= set(T.name):
    t, s = T.set_index("name").loc["E1_enc_full"], T.set_index("name").loc["E6_enc_half_kd"]; print(f"\nDISTILLATION: teacher E1 {t['params_M']:.1f}M params, dev F1 {t['dev_f1']:.4f}, evaluation {t['eval_minutes']:.1f} min | student {s['params_M']:.1f}M params ({100 * s['params_M'] / t['params_M']:.0f}%), dev F1 {s['dev_f1']:.4f}, evaluation {s['eval_minutes']:.1f} min")
if len(T): (WORK / "analysis.md").write_text("# Phase 7 results (dev, not official)\n\n```\n" + (COMP.to_string(index=False) if "COMP" in globals() else T.to_string()) + "\n```\n", encoding="utf-8")


In [ ]:
# ================= CELL 19 — OUTPUT CHECK  [RUNS NOW] =================
shutil.rmtree(WORK / "hf", ignore_errors=True)
for p in (PRED_PATH, PRED_PATH.with_name("prediction_hybrid.json")):
    if p.exists(): validate_prediction(p, DATA["test"]); sub_ = json.load(open(p)); print(f"{p.name}: VALID ({len(sub_)} test queries x {TOP_K} unique candidates from the test pool) | zip {p.with_suffix('.zip').name} exists: {p.with_suffix('.zip').exists()} | first entry: {list(sub_.items())[0]}")
print("ledger:", LEDGER, "| runs:", sorted(p.name for p in (WORK / "runs").iterdir()))
